# E_S5 — Power-law shape analysis after Strong monotonicity

This notebook starts from the Strong monotonicity cases selected by E_S4, retrieves the original scatter points from E_S3, and tests whether a single power-law model can distinguish linear, concave, convex, and other strong-monotonic shapes. The current run uses the **strong variant only across all retained noise levels**; mild and standard are disabled in the switches at the top.

In [1]:
# ============================================================
# INPUT / OUTPUT PATHS — EDIT THESE VALUES FIRST
# ============================================================
S3_DATA_DIR = 'E/output/S3_snr_mic_sweep'
S4_DATA_DIR = 'E/output/S4_mic_range_filter'
SCATTER_POINTS_FILENAME = 'scatter_points.npz'
OUTPUT_DATA_DIR = 'E/output/S5_power_law_shape_analysis'

# The active hierarchy is configured locally at the beginning of each step.
# Power law confirms Linear. Every remaining Simple, non-linear case then
# receives one Cubic fit for the final curvature decision.

## Load the E_S4 parent cohort and E_S3 scatter points

In [2]:
from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from scipy.optimize import OptimizeWarning, curve_fit
from scipy.stats import norm
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold


def locate_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / 'E').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate repository root containing E/.')


def resolve_configured_path(value: str) -> Path:
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (REPO_ROOT / path).resolve()


REPO_ROOT = locate_repo_root()
S3_DIR = resolve_configured_path(S3_DATA_DIR)
S4_DIR = resolve_configured_path(S4_DATA_DIR)
SCATTER_POINTS_PATH = S3_DIR / SCATTER_POINTS_FILENAME
OUTPUT_DIR = resolve_configured_path(OUTPUT_DATA_DIR)

for required_path in [SCATTER_POINTS_PATH, S4_DIR]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'S4 data:   {S4_DIR}')
print(f'S3 points: {SCATTER_POINTS_PATH}')
print(f'Output:    {OUTPUT_DIR}')

S4 data:   /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/E/output/S4_mic_range_filter
S3 points: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/E/output/S3_snr_mic_sweep/scatter_points.npz
Output:    /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/E/output/S5_power_law_shape_analysis


In [3]:
%%script true
strong_monotonicity_cases = pd.read_parquet(STRONG_MONOTONICITY_PATH)
required_columns = {
    'candidate_index', 'case_id', 'family_id', 'family_name',
    'variant_level', 'repeat', 'inverse_snr', 'MIC',
    'pearson_r', 'spearman_rho',
}
missing_columns = required_columns.difference(strong_monotonicity_cases.columns)
if missing_columns:
    raise KeyError(f'Missing S4 columns: {sorted(missing_columns)}')
if strong_monotonicity_cases['candidate_index'].duplicated().any():
    raise ValueError('S4 Strong monotonicity data contain duplicate candidate_index values.')

analysis_cases = strong_monotonicity_cases.copy()
variant_switches = {
    'mild': INCLUDE_VARIANT_MILD,
    'standard': INCLUDE_VARIANT_STANDARD,
    'strong': INCLUDE_VARIANT_STRONG,
}
enabled_variants = [name for name, enabled in variant_switches.items() if enabled]
if not enabled_variants:
    raise ValueError('At least one variant switch must be enabled.')
analysis_cases = analysis_cases.loc[
    analysis_cases['variant_level'].isin(enabled_variants)
].copy()
if NOISE_FREE_PILOT_ONLY:
    analysis_cases = analysis_cases.loc[
        analysis_cases['inverse_snr'].abs() <= NOISE_FREE_TOLERANCE
    ].copy()
analysis_cases = analysis_cases.sort_values('candidate_index').reset_index(drop=True)
analysis_cases.insert(0, 'pilot_row', np.arange(len(analysis_cases), dtype=int))

points = np.load(SCATTER_POINTS_PATH)
if set(points.files) != {'x', 'y'}:
    raise KeyError(f'Expected x and y arrays, found {points.files}.')
if points['x'].shape != points['y'].shape:
    raise ValueError('x and y arrays have different shapes.')
candidate_indices = analysis_cases['candidate_index'].to_numpy(dtype=int)
if candidate_indices.min() < 0 or candidate_indices.max() >= points['x'].shape[0]:
    raise IndexError('candidate_index falls outside scatter_points.npz.')

analysis_x = points['x'][candidate_indices].astype(float)
analysis_y = points['y'][candidate_indices].astype(float)
if analysis_x.shape[1] < 10:
    raise ValueError('Too few points per scatterplot for power-law fitting.')
if not np.isfinite(analysis_x).all() or not np.isfinite(analysis_y).all():
    raise ValueError('Non-finite scatter points found.')
if (analysis_x <= 0).any():
    raise ValueError('Power-law fitting requires positive x values.')

print(f'S4 Strong monotonicity parent: {len(strong_monotonicity_cases):,}')
print(f'Enabled variants:              {enabled_variants}')
print(f'Cases in this run:             {len(analysis_cases):,}')
display(
    analysis_cases.groupby(['family_id', 'variant_level'])
    .size().unstack(fill_value=0)
)

## Legacy comparison blocks — disabled

These historical comparison cells are retained only for provenance and are skipped during execution. They are not part of the current classifier.

The active hierarchy below uses only the strict free power-law model $y=ax^b+c$. No cubic-polynomial derivative pattern is used.

In [4]:
%%script true
def power_law(x: np.ndarray, a: float, b: float, c: float) -> np.ndarray:
    return c + a * np.power(x, b)


def initial_power_parameters(x: np.ndarray, y: np.ndarray) -> tuple:
    y_low = float(np.quantile(y, 0.05))
    y_high = float(np.quantile(y, 0.95))
    amplitude = max(y_high - y_low, np.finfo(float).eps)
    return amplitude, 1.0, y_low


def fit_power_parameters(x: np.ndarray, y: np.ndarray):
    p0 = initial_power_parameters(x, y)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', OptimizeWarning)
        parameters, covariance = curve_fit(
            power_law,
            x,
            y,
            p0=p0,
            bounds=(
                [0.0, POWER_EXPONENT_MIN, -np.inf],
                [np.inf, POWER_EXPONENT_MAX, np.inf],
            ),
            maxfev=30000,
        )
    return parameters, covariance


def normalized_rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    scale = float(np.std(y_true, ddof=1))
    if not np.isfinite(scale) or scale <= np.finfo(float).eps:
        return np.nan
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)) / scale)


def information_criterion(y_true: np.ndarray, y_pred: np.ndarray, k: int) -> float:
    n = len(y_true)
    rss = max(float(np.sum((y_true - y_pred) ** 2)), np.finfo(float).tiny)
    return float(n * np.log(rss / n) + k * np.log(n))


def signed_absolute_mass(values: np.ndarray) -> dict:
    positive_mass = float(np.abs(values[values > 0]).sum())
    negative_mass = float(np.abs(values[values < 0]).sum())
    total_mass = positive_mass + negative_mass
    if total_mass <= np.finfo(float).eps:
        return {'q_positive': 0.5, 'q_negative': 0.5, 'dominance': 0.5, 'dominant_sign': 0}
    q_positive = positive_mass / total_mass
    q_negative = negative_mass / total_mass
    return {
        'q_positive': q_positive,
        'q_negative': q_negative,
        'dominance': max(q_positive, q_negative),
        'dominant_sign': 1 if q_positive >= q_negative else -1,
    }


def derivative_pattern_features(x: np.ndarray, y: np.ndarray) -> dict:
    order = np.argsort(x)
    x_sorted = x[order]
    y_sorted = y[order]
    x_scale = float(np.std(x_sorted, ddof=1))
    y_scale = float(np.std(y_sorted, ddof=1))
    if x_scale <= np.finfo(float).eps or y_scale <= np.finfo(float).eps:
        raise ValueError('Derivative pattern requires non-constant x and y.')
    x_standardized = (x_sorted - x_sorted.mean()) / x_scale
    y_standardized = (y_sorted - y_sorted.mean()) / y_scale
    polynomial = np.poly1d(np.polyfit(x_standardized, y_standardized, deg=3))
    x_lower, x_upper = np.quantile(
        x_standardized,
        [DERIVATIVE_INTERIOR_MARGIN, 1.0 - DERIVATIVE_INTERIOR_MARGIN],
    )
    grid = np.linspace(x_lower, x_upper, DERIVATIVE_GRID_SIZE)
    first_derivative = np.polyder(polynomial, 1)(grid)
    second_derivative = np.polyder(polynomial, 2)(grid)
    first = signed_absolute_mass(first_derivative)
    second = signed_absolute_mass(second_derivative)
    return {
        'f1_q_positive': first['q_positive'],
        'f1_q_negative': first['q_negative'],
        'f1_dominance': first['dominance'],
        'f1_dominant_sign': first['dominant_sign'],
        'f2_q_positive': second['q_positive'],
        'f2_q_negative': second['q_negative'],
        'f2_dominance': second['dominance'],
        'f2_dominant_sign': second['dominant_sign'],
        'curvature_strength': float(
            np.mean(np.abs(second_derivative))
            / (np.mean(np.abs(first_derivative)) + np.finfo(float).eps)
        ),
    }


def residual_runs_features(x: np.ndarray, y: np.ndarray, predicted: np.ndarray) -> dict:
    residuals = (y - predicted)[np.argsort(x)]
    signs = np.sign(residuals)
    signs = signs[signs != 0]
    n_positive = int(np.sum(signs > 0))
    n_negative = int(np.sum(signs < 0))
    n = n_positive + n_negative
    if n_positive == 0 or n_negative == 0 or n < 2:
        runs = 1 if n else 0
        runs_z = np.nan
        runs_p_lower = np.nan
    else:
        runs = int(1 + np.sum(signs[1:] != signs[:-1]))
        expected_runs = 1.0 + 2.0 * n_positive * n_negative / n
        runs_variance = (
            2.0 * n_positive * n_negative
            * (2.0 * n_positive * n_negative - n)
            / (n ** 2 * (n - 1))
        )
        runs_z = (
            (runs - expected_runs) / np.sqrt(runs_variance)
            if runs_variance > 0 else np.nan
        )
        runs_p_lower = float(norm.cdf(runs_z)) if np.isfinite(runs_z) else np.nan
    y_range = float(np.ptp(y))
    relative_rmse = (
        float(np.sqrt(np.mean(residuals ** 2)) / y_range)
        if y_range > np.finfo(float).eps else np.nan
    )
    return {
        'runs_count': runs,
        'runs_z': runs_z,
        'runs_p_lower': runs_p_lower,
        'residual_relative_rmse_range': relative_rmse,
    }


def fit_one_scatterplot(
    x: np.ndarray, y: np.ndarray, candidate_index: int
) -> dict:
    result = {
        'fit_converged': False,
        'fit_error': '',
        'a': np.nan,
        'b': np.nan,
        'c': np.nan,
        'b_se': np.nan,
        'b_ci_lower': np.nan,
        'b_ci_upper': np.nan,
        'r2': np.nan,
        'cv_r2': np.nan,
        'nrmse': np.nan,
        'power_bic': np.nan,
        'linear_bic': np.nan,
        'delta_bic_linear_minus_power': np.nan,
        'f1_q_positive': np.nan,
        'f1_q_negative': np.nan,
        'f1_dominance': np.nan,
        'f1_dominant_sign': 0,
        'f2_q_positive': np.nan,
        'f2_q_negative': np.nan,
        'f2_dominance': np.nan,
        'f2_dominant_sign': 0,
        'curvature_strength': np.nan,
        'runs_count': np.nan,
        'runs_z': np.nan,
        'runs_p_lower': np.nan,
        'residual_relative_rmse_range': np.nan,
    }
    try:
        parameters, covariance = fit_power_parameters(x, y)
        a, b, c = map(float, parameters)
        predicted = power_law(x, a, b, c)

        b_variance = float(covariance[1, 1]) if covariance.shape == (3, 3) else np.nan
        b_se = np.sqrt(b_variance) if np.isfinite(b_variance) and b_variance >= 0 else np.nan
        z_value = float(norm.ppf(0.5 + B_CONFIDENCE_LEVEL / 2.0))
        b_ci_lower = b - z_value * b_se if np.isfinite(b_se) else np.nan
        b_ci_upper = b + z_value * b_se if np.isfinite(b_se) else np.nan

        cv = KFold(
            n_splits=CV_FOLDS,
            shuffle=True,
            random_state=RANDOM_SEED + int(candidate_index),
        )
        held_out_observed = []
        held_out_predicted = []
        for train_index, test_index in cv.split(x):
            fold_parameters, _ = fit_power_parameters(x[train_index], y[train_index])
            fold_prediction = power_law(x[test_index], *fold_parameters)
            held_out_observed.append(y[test_index])
            held_out_predicted.append(fold_prediction)
        cv_observed = np.concatenate(held_out_observed)
        cv_predicted = np.concatenate(held_out_predicted)

        linear_design = np.column_stack([x, np.ones_like(x)])
        linear_coefficients, *_ = np.linalg.lstsq(linear_design, y, rcond=None)
        linear_prediction = linear_design @ linear_coefficients
        power_bic = information_criterion(y, predicted, k=3)
        linear_bic = information_criterion(y, linear_prediction, k=2)

        result.update({
            'fit_converged': True,
            'a': a,
            'b': b,
            'c': c,
            'b_se': b_se,
            'b_ci_lower': b_ci_lower,
            'b_ci_upper': b_ci_upper,
            'r2': float(r2_score(y, predicted)),
            'cv_r2': float(r2_score(cv_observed, cv_predicted)),
            'nrmse': normalized_rmse(y, predicted),
            'power_bic': power_bic,
            'linear_bic': linear_bic,
            'delta_bic_linear_minus_power': linear_bic - power_bic,
            **derivative_pattern_features(x, y),
            **residual_runs_features(x, y, predicted),
        })
    except Exception as error:
        result['fit_error'] = f'{type(error).__name__}: {error}'
    return result


In [5]:
%%script true
fit_rows = []
for row_index, case in analysis_cases.iterrows():
    fit_result = fit_one_scatterplot(
        analysis_x[row_index],
        analysis_y[row_index],
        int(case['candidate_index']),
    )
    fit_rows.append(fit_result)

fit_results = pd.concat(
    [analysis_cases.reset_index(drop=True), pd.DataFrame(fit_rows)],
    axis=1,
)
fit_results['fit_acceptable'] = (
    fit_results['fit_converged']
    & fit_results['cv_r2'].ge(MIN_ACCEPTABLE_CV_R2)
    & fit_results['nrmse'].le(MAX_ACCEPTABLE_NRMSE)
)

fit_results['abs_b_minus_one'] = (fit_results['b'] - 1.0).abs()
fit_results['bic_evidence'] = 'Intermediate evidence'
fit_results.loc[
    fit_results['delta_bic_linear_minus_power'].le(DELTA_BIC_WEAK_THRESHOLD),
    'bic_evidence',
] = 'No meaningful power-law advantage'
fit_results.loc[
    fit_results['delta_bic_linear_minus_power'].ge(DELTA_BIC_STRONG_THRESHOLD),
    'bic_evidence',
] = 'Strong power-law evidence'

near_linear_exponent = fit_results['abs_b_minus_one'].le(
    POWER_EXPONENT_LINEAR_TOLERANCE
)
below_linear_exponent = fit_results['b'].lt(
    1.0 - POWER_EXPONENT_LINEAR_TOLERANCE
)
above_linear_exponent = fit_results['b'].gt(
    1.0 + POWER_EXPONENT_LINEAR_TOLERANCE
)

# Method 1: b only.
converged = fit_results['fit_converged']
fit_results['shape_b_only'] = 'Unclassified'
fit_results.loc[converged & near_linear_exponent, 'shape_b_only'] = 'Linear'
fit_results.loc[converged & below_linear_exponent, 'shape_b_only'] = 'Concave'
fit_results.loc[converged & above_linear_exponent, 'shape_b_only'] = 'Convex'

# Method 2 (PRIMARY): b + first/second derivative pattern.
stable_first_derivative = fit_results['f1_dominance'].ge(
    FIRST_DERIVATIVE_DOMINANCE_THRESHOLD
)
stable_second_derivative = fit_results['f2_dominance'].ge(
    SECOND_DERIVATIVE_DOMINANCE_THRESHOLD
)
nonlinear_exponent = ~near_linear_exponent
fit_results['shape_b_derivative'] = 'Uncertain'
fit_results.loc[~converged, 'shape_b_derivative'] = 'Unclassified'
fit_results.loc[
    converged & near_linear_exponent,
    'shape_b_derivative',
] = 'Linear'
fit_results.loc[
    converged & nonlinear_exponent & stable_first_derivative
    & stable_second_derivative & fit_results['f2_dominant_sign'].gt(0),
    'shape_b_derivative',
] = 'Convex'
fit_results.loc[
    converged & nonlinear_exponent & stable_first_derivative
    & stable_second_derivative & fit_results['f2_dominant_sign'].lt(0),
    'shape_b_derivative',
] = 'Concave'
fit_results.loc[
    converged & nonlinear_exponent & stable_first_derivative
    & ~stable_second_derivative,
    'shape_b_derivative',
] = 'Other strong-monotonic'

# Method 3: b + one-sided Wald–Wolfowitz Runs test on ordered residual signs.
fit_results['runs_structured_residual'] = (
    converged
    & fit_results['runs_z'].lt(RUNS_Z_THRESHOLD)
    & fit_results['residual_relative_rmse_range'].gt(
        RUNS_NUMERICAL_RELATIVE_RMSE_TOLERANCE
    )
)
fit_results['shape_b_runs'] = fit_results['shape_b_only']
fit_results.loc[
    fit_results['runs_structured_residual'], 'shape_b_runs'
] = 'Other strong-monotonic'

# Method 4: b + ordinary in-sample R² adequacy.
fit_results['shape_b_r2'] = fit_results['shape_b_only']
fit_results.loc[
    converged & fit_results['r2'].lt(R2_COMPARISON_THRESHOLD),
    'shape_b_r2',
] = 'Other strong-monotonic'

# Main S5 label: derivative-pattern method.
fit_results['power_law_shape'] = fit_results['shape_b_derivative']

# Known synthetic family labels are used only to evaluate accuracy.
fit_results['expected_power_law_shape'] = fit_results['family_id'].map(
    EXPECTED_SHAPE_BY_FAMILY
)
fit_results['shape_evaluable'] = fit_results['expected_power_law_shape'].notna()
METHOD_SPECS = {
    'b only': ('shape_b_only', 'correct_b_only'),
    'b + derivatives (primary)': ('shape_b_derivative', 'correct_b_derivative'),
    'b + Runs test': ('shape_b_runs', 'correct_b_runs'),
    'b + R²': ('shape_b_r2', 'correct_b_r2'),
}
for method, (shape_column, correct_column) in METHOD_SPECS.items():
    fit_results[correct_column] = (
        fit_results['shape_evaluable']
        & fit_results[shape_column].eq(fit_results['expected_power_law_shape'])
    )
fit_results['shape_correct'] = fit_results['correct_b_derivative']

evaluated_results = fit_results.loc[fit_results['shape_evaluable']].copy()
accuracy_by_noise_level = (
    evaluated_results.groupby('inverse_snr', as_index=False)
    .agg(
        n_cases=('shape_correct', 'size'),
        n_correct=('shape_correct', 'sum'),
        accuracy=('shape_correct', 'mean'),
    )
    .sort_values('inverse_snr')
    .reset_index(drop=True)
)
accuracy_by_noise_level['retention_share_of_max'] = (
    accuracy_by_noise_level['n_cases'] / accuracy_by_noise_level['n_cases'].max()
)

family_accuracy_by_noise_level = (
    evaluated_results.groupby(
        ['family_id', 'family_name', 'inverse_snr'], as_index=False
    )
    .agg(
        n_cases=('shape_correct', 'size'),
        n_correct=('shape_correct', 'sum'),
        accuracy=('shape_correct', 'mean'),
    )
    .sort_values(['family_id', 'inverse_snr'])
    .reset_index(drop=True)
)

cumulative_rows = []
for max_inverse_snr in accuracy_by_noise_level['inverse_snr']:
    cumulative = evaluated_results.loc[
        evaluated_results['inverse_snr'].le(max_inverse_snr)
    ]
    cumulative_rows.append({
        'max_inverse_snr': float(max_inverse_snr),
        'n_cases': len(cumulative),
        'n_correct': int(cumulative['shape_correct'].sum()),
        'cumulative_accuracy': float(cumulative['shape_correct'].mean()),
    })
cumulative_accuracy_by_noise = pd.DataFrame(cumulative_rows)

method_accuracy_rows = []
method_family_rows = []
method_noise_rows = []
method_cumulative_rows = []
noise_levels = np.sort(evaluated_results['inverse_snr'].unique())
for method, (_, correct_column) in METHOD_SPECS.items():
    method_accuracy_rows.append({
        'method': method,
        'n_cases': len(evaluated_results),
        'n_correct': int(evaluated_results[correct_column].sum()),
        'accuracy': float(evaluated_results[correct_column].mean()),
    })
    family_table = (
        evaluated_results.groupby(['family_id', 'family_name'], as_index=False)
        .agg(
            n_cases=(correct_column, 'size'),
            n_correct=(correct_column, 'sum'),
            accuracy=(correct_column, 'mean'),
        )
    )
    family_table.insert(0, 'method', method)
    method_family_rows.append(family_table)
    noise_table = (
        evaluated_results.groupby('inverse_snr', as_index=False)
        .agg(
            n_cases=(correct_column, 'size'),
            n_correct=(correct_column, 'sum'),
            accuracy=(correct_column, 'mean'),
        )
    )
    noise_table.insert(0, 'method', method)
    method_noise_rows.append(noise_table)
    for max_inverse_snr in noise_levels:
        cumulative = evaluated_results.loc[
            evaluated_results['inverse_snr'].le(max_inverse_snr)
        ]
        method_cumulative_rows.append({
            'method': method,
            'max_inverse_snr': float(max_inverse_snr),
            'n_cases': len(cumulative),
            'n_correct': int(cumulative[correct_column].sum()),
            'cumulative_accuracy': float(cumulative[correct_column].mean()),
        })
method_accuracy_summary = pd.DataFrame(method_accuracy_rows)
method_family_accuracy = pd.concat(method_family_rows, ignore_index=True)
method_accuracy_by_noise = pd.concat(method_noise_rows, ignore_index=True)
method_cumulative_accuracy = pd.DataFrame(method_cumulative_rows)

print('Fit convergence:')
print(fit_results['fit_converged'].value_counts(dropna=False).to_string())
print('\nLegacy CV R² + NRMSE diagnostic (not used for routing):')
print(fit_results['fit_acceptable'].value_counts(dropna=False).to_string())
print('\nPrimary derivative-pattern labels:')
print(fit_results['power_law_shape'].value_counts().to_string())
print('\nFour-method accuracy comparison:')
display(
    method_accuracy_summary.assign(
        accuracy=method_accuracy_summary['accuracy'].map('{:.2%}'.format)
    )
)
print('\nPrimary accuracy by family:')
display(
    method_family_accuracy.loc[
        method_family_accuracy['method'].eq('b + derivatives (primary)')
    ].assign(
        accuracy=lambda table: table['accuracy'].map('{:.2%}'.format)
    )
)

## Mild vs Standard vs Strong comparison

This additional block applies the same four classifiers to the S4-retained `mild`, `standard`, and `strong` cohorts. Accuracy is evaluated only for F01 Linear, F03 Convex, F05 Concave, and F13 S-curve because these are the families with an explicit expected label in `EXPECTED_SHAPE_BY_FAMILY`. F07, F15, and F23 remain unevaluated rather than being assigned an artificial ground-truth class.

In [6]:
%%script true
COMPARISON_VARIANTS = ['mild', 'standard', 'strong']
KNOWN_SHAPE_FAMILIES = list(EXPECTED_SHAPE_BY_FAMILY)


def add_four_classifier_labels(results: pd.DataFrame) -> pd.DataFrame:
    results = results.copy()
    results['abs_b_minus_one'] = (results['b'] - 1.0).abs()
    converged = results['fit_converged']
    near_linear = results['abs_b_minus_one'].le(POWER_EXPONENT_LINEAR_TOLERANCE)
    below_linear = results['b'].lt(1.0 - POWER_EXPONENT_LINEAR_TOLERANCE)
    above_linear = results['b'].gt(1.0 + POWER_EXPONENT_LINEAR_TOLERANCE)

    results['shape_b_only'] = 'Unclassified'
    results.loc[converged & near_linear, 'shape_b_only'] = 'Linear'
    results.loc[converged & below_linear, 'shape_b_only'] = 'Concave'
    results.loc[converged & above_linear, 'shape_b_only'] = 'Convex'

    stable_f1 = results['f1_dominance'].ge(FIRST_DERIVATIVE_DOMINANCE_THRESHOLD)
    stable_f2 = results['f2_dominance'].ge(SECOND_DERIVATIVE_DOMINANCE_THRESHOLD)
    results['shape_b_derivative'] = 'Uncertain'
    results.loc[~converged, 'shape_b_derivative'] = 'Unclassified'
    results.loc[converged & near_linear, 'shape_b_derivative'] = 'Linear'
    results.loc[
        converged & ~near_linear & stable_f1 & stable_f2
        & results['f2_dominant_sign'].gt(0),
        'shape_b_derivative',
    ] = 'Convex'
    results.loc[
        converged & ~near_linear & stable_f1 & stable_f2
        & results['f2_dominant_sign'].lt(0),
        'shape_b_derivative',
    ] = 'Concave'
    results.loc[
        converged & ~near_linear & stable_f1 & ~stable_f2,
        'shape_b_derivative',
    ] = 'Other strong-monotonic'

    results['runs_structured_residual'] = (
        converged
        & results['runs_z'].lt(RUNS_Z_THRESHOLD)
        & results['residual_relative_rmse_range'].gt(
            RUNS_NUMERICAL_RELATIVE_RMSE_TOLERANCE
        )
    )
    results['shape_b_runs'] = results['shape_b_only']
    results.loc[
        results['runs_structured_residual'], 'shape_b_runs'
    ] = 'Other strong-monotonic'

    results['shape_b_r2'] = results['shape_b_only']
    results.loc[
        converged & results['r2'].lt(R2_COMPARISON_THRESHOLD),
        'shape_b_r2',
    ] = 'Other strong-monotonic'
    results['expected_power_law_shape'] = results['family_id'].map(
        EXPECTED_SHAPE_BY_FAMILY
    )
    for _, (shape_column, correct_column) in METHOD_SPECS.items():
        results[correct_column] = results[shape_column].eq(
            results['expected_power_law_shape']
        )
    return results


variant_fit_frames = []
for variant in COMPARISON_VARIANTS:
    variant_cases = strong_monotonicity_cases.loc[
        strong_monotonicity_cases['variant_level'].eq(variant)
        & strong_monotonicity_cases['family_id'].isin(KNOWN_SHAPE_FAMILIES)
    ].sort_values('candidate_index').reset_index(drop=True)
    if variant == 'strong':
        variant_results = fit_results.loc[
            fit_results['family_id'].isin(KNOWN_SHAPE_FAMILIES)
        ].copy()
    else:
        print(f'Fitting {variant}: {len(variant_cases):,} S4-retained cases')
        variant_indices = variant_cases['candidate_index'].to_numpy(dtype=int)
        variant_x = points['x'][variant_indices].astype(float)
        variant_y = points['y'][variant_indices].astype(float)
        variant_fit_rows = [
            fit_one_scatterplot(
                variant_x[row_index], variant_y[row_index],
                int(case['candidate_index']),
            )
            for row_index, case in variant_cases.iterrows()
        ]
        variant_results = pd.concat(
            [variant_cases, pd.DataFrame(variant_fit_rows)], axis=1
        )
    variant_fit_frames.append(add_four_classifier_labels(variant_results))

variant_comparison_fits = pd.concat(variant_fit_frames, ignore_index=True)
variant_method_rows = []
variant_family_rows = []
for method, (_, correct_column) in METHOD_SPECS.items():
    overall = (
        variant_comparison_fits.groupby('variant_level', as_index=False)
        .agg(
            n_cases=(correct_column, 'size'),
            n_correct=(correct_column, 'sum'),
            accuracy=(correct_column, 'mean'),
        )
    )
    overall.insert(1, 'method', method)
    variant_method_rows.append(overall)
    by_family = (
        variant_comparison_fits.groupby(
            ['variant_level', 'family_id', 'family_name'], as_index=False
        )
        .agg(
            n_cases=(correct_column, 'size'),
            n_correct=(correct_column, 'sum'),
            accuracy=(correct_column, 'mean'),
        )
    )
    by_family.insert(1, 'method', method)
    variant_family_rows.append(by_family)

variant_method_accuracy = pd.concat(variant_method_rows, ignore_index=True)
variant_family_accuracy = pd.concat(variant_family_rows, ignore_index=True)
variant_method_accuracy['variant_level'] = pd.Categorical(
    variant_method_accuracy['variant_level'], COMPARISON_VARIANTS, ordered=True
)
variant_family_accuracy['variant_level'] = pd.Categorical(
    variant_family_accuracy['variant_level'], COMPARISON_VARIANTS, ordered=True
)
variant_method_accuracy = variant_method_accuracy.sort_values(
    ['variant_level', 'method']
)
variant_family_accuracy = variant_family_accuracy.sort_values(
    ['variant_level', 'family_id', 'method']
)

variant_fits_path = OUTPUT_DIR / 'mild_standard_strong_classifier_comparison_fits.parquet'
variant_method_path = OUTPUT_DIR / 'mild_standard_strong_method_accuracy.csv'
variant_family_path = OUTPUT_DIR / 'mild_standard_strong_family_accuracy.csv'
variant_comparison_fits.to_parquet(variant_fits_path, index=False)
variant_method_accuracy.to_csv(variant_method_path, index=False)
variant_family_accuracy.to_csv(variant_family_path, index=False)

VARIANT_COLORS = {'mild': '#56B4E9', 'standard': '#E69F00', 'strong': '#009E73'}
METHOD_COLORS_VARIANT = {
    'b only': '#7A7A7A',
    'b + derivatives (primary)': '#009E73',
    'b + Runs test': '#D55E00',
    'b + R²': '#7B61A8',
}
method_order_variant = list(METHOD_SPECS)
fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.8))
variant_positions = np.arange(len(COMPARISON_VARIANTS))
method_width = 0.19
for method_index, method in enumerate(method_order_variant):
    values = (
        variant_method_accuracy.loc[
            variant_method_accuracy['method'].eq(method)
        ].set_index('variant_level').reindex(COMPARISON_VARIANTS)['accuracy']
    )
    axes[0].bar(
        variant_positions + (method_index - 1.5) * method_width,
        values, width=method_width,
        color=METHOD_COLORS_VARIANT[method], label=method,
    )
axes[0].set_xticks(variant_positions)
axes[0].set_xticklabels([name.title() for name in COMPARISON_VARIANTS])
axes[0].set_ylim(0.0, 1.04)
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Overall accuracy by variant and method')
axes[0].grid(True, axis='y', alpha=0.22)

primary_family = variant_family_accuracy.loc[
    variant_family_accuracy['method'].eq('b + derivatives (primary)')
]
family_ids_variant = KNOWN_SHAPE_FAMILIES
family_positions = np.arange(len(family_ids_variant))
variant_width = 0.25
for variant_index, variant in enumerate(COMPARISON_VARIANTS):
    values = (
        primary_family.loc[
            primary_family['variant_level'].eq(variant)
        ].set_index('family_id').reindex(family_ids_variant)['accuracy']
    )
    axes[1].bar(
        family_positions + (variant_index - 1) * variant_width,
        values, width=variant_width,
        color=VARIANT_COLORS[variant], label=variant.title(),
    )
axes[1].set_xticks(family_positions)
axes[1].set_xticklabels(family_ids_variant)
axes[1].set_ylim(0.0, 1.04)
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Primary derivative method by family')
axes[1].grid(True, axis='y', alpha=0.22)
axes[1].legend(loc='lower right')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4, frameon=True)
fig.suptitle('S4-retained cases — Mild vs Standard vs Strong', fontsize=15)
fig.tight_layout(rect=[0, 0.10, 1, 0.94])
variant_figure_path = FIGURE_DIR / 'mild_standard_strong_accuracy_comparison.png'
fig.savefig(variant_figure_path, dpi=FIGURE_DPI, bbox_inches='tight')
plt.close(fig)

print('Overall comparison:')
display(
    variant_method_accuracy.assign(
        accuracy=variant_method_accuracy['accuracy'].map('{:.2%}'.format)
    )
)
print(f'Saved: {variant_figure_path.relative_to(REPO_ROOT)}')
display(Image(filename=str(variant_figure_path)))

## Full S4-retained cohorts: coarse shape-category accuracy

The following blocks include **every function family retained by S4**, in the requested order: Strong first, then Mild, then Standard. Accuracy here is deliberately a coarse S5 shape-category accuracy, not exact family identification: F07 Saturation is expected to be `Concave`, while F13 S-curve, F15 Threshold, and F23 Two Lines are grouped as `Other strong-monotonic`. Consequently, this section does not claim to distinguish S-curve from Threshold.

In [7]:
%%script true
# Block 1 — Strong: all function families that passed S4.
FULL_S4_COARSE_EXPECTED_SHAPE = {
    'F01': 'Linear',
    'F03': 'Convex',
    'F05': 'Concave',
    'F07': 'Concave',
    'F13': 'Other strong-monotonic',
    'F15': 'Other strong-monotonic',
    'F23': 'Other strong-monotonic',
}


def attach_coarse_ground_truth(results: pd.DataFrame) -> pd.DataFrame:
    results = results.copy()
    results['expected_coarse_shape'] = results['family_id'].map(
        FULL_S4_COARSE_EXPECTED_SHAPE
    )
    if results['expected_coarse_shape'].isna().any():
        missing = sorted(
            results.loc[results['expected_coarse_shape'].isna(), 'family_id'].unique()
        )
        raise KeyError(f'Missing coarse expected shape for: {missing}')
    for _, (shape_column, correct_column) in METHOD_SPECS.items():
        results[f'coarse_{correct_column}'] = results[shape_column].eq(
            results['expected_coarse_shape']
        )
    return results


def summarize_coarse_methods(results: pd.DataFrame) -> pd.DataFrame:
    rows = []
    variant = str(results['variant_level'].iloc[0])
    for method, (_, correct_column) in METHOD_SPECS.items():
        coarse_correct = f'coarse_{correct_column}'
        rows.append({
            'variant_level': variant,
            'method': method,
            'n_cases': len(results),
            'n_correct': int(results[coarse_correct].sum()),
            'accuracy': float(results[coarse_correct].mean()),
        })
    return pd.DataFrame(rows)


strong_all_s4_results = attach_coarse_ground_truth(
    add_four_classifier_labels(fit_results.copy())
)
strong_all_s4_accuracy = summarize_coarse_methods(strong_all_s4_results)
print('Strong — all S4-retained function families:')
display(strong_all_s4_results.groupby('family_id').size().rename('n_cases').to_frame())
display(
    strong_all_s4_accuracy.assign(
        accuracy=strong_all_s4_accuracy['accuracy'].map('{:.2%}'.format)
    )
)

In [8]:
%%script true
# Block 2 — Mild: reuse the four previously fitted families and fit every additional S4 family.
mild_known_results = variant_comparison_fits.loc[
    variant_comparison_fits['variant_level'].eq('mild')
].copy()
mild_extra_cases = strong_monotonicity_cases.loc[
    strong_monotonicity_cases['variant_level'].eq('mild')
    & ~strong_monotonicity_cases['family_id'].isin(KNOWN_SHAPE_FAMILIES)
].sort_values('candidate_index').reset_index(drop=True)
print(f'Fitting Mild additional families: {len(mild_extra_cases):,} cases')
mild_extra_indices = mild_extra_cases['candidate_index'].to_numpy(dtype=int)
mild_extra_x = points['x'][mild_extra_indices].astype(float)
mild_extra_y = points['y'][mild_extra_indices].astype(float)
mild_extra_fit_rows = [
    fit_one_scatterplot(
        mild_extra_x[row_index], mild_extra_y[row_index],
        int(case['candidate_index']),
    )
    for row_index, case in mild_extra_cases.iterrows()
]
mild_extra_results = add_four_classifier_labels(
    pd.concat([mild_extra_cases, pd.DataFrame(mild_extra_fit_rows)], axis=1)
)
mild_all_s4_results = attach_coarse_ground_truth(
    pd.concat([mild_known_results, mild_extra_results], ignore_index=True)
)
mild_all_s4_accuracy = summarize_coarse_methods(mild_all_s4_results)
print('Mild — all S4-retained function families:')
display(mild_all_s4_results.groupby('family_id').size().rename('n_cases').to_frame())
display(
    mild_all_s4_accuracy.assign(
        accuracy=mild_all_s4_accuracy['accuracy'].map('{:.2%}'.format)
    )
)

In [9]:
%%script true
# Block 3 — Standard: reuse the four previously fitted families and fit every additional S4 family.
standard_known_results = variant_comparison_fits.loc[
    variant_comparison_fits['variant_level'].eq('standard')
].copy()
standard_extra_cases = strong_monotonicity_cases.loc[
    strong_monotonicity_cases['variant_level'].eq('standard')
    & ~strong_monotonicity_cases['family_id'].isin(KNOWN_SHAPE_FAMILIES)
].sort_values('candidate_index').reset_index(drop=True)
print(f'Fitting Standard additional families: {len(standard_extra_cases):,} cases')
standard_extra_indices = standard_extra_cases['candidate_index'].to_numpy(dtype=int)
standard_extra_x = points['x'][standard_extra_indices].astype(float)
standard_extra_y = points['y'][standard_extra_indices].astype(float)
standard_extra_fit_rows = [
    fit_one_scatterplot(
        standard_extra_x[row_index], standard_extra_y[row_index],
        int(case['candidate_index']),
    )
    for row_index, case in standard_extra_cases.iterrows()
]
standard_extra_results = add_four_classifier_labels(
    pd.concat([standard_extra_cases, pd.DataFrame(standard_extra_fit_rows)], axis=1)
)
standard_all_s4_results = attach_coarse_ground_truth(
    pd.concat([standard_known_results, standard_extra_results], ignore_index=True)
)
standard_all_s4_accuracy = summarize_coarse_methods(standard_all_s4_results)
print('Standard — all S4-retained function families:')
display(standard_all_s4_results.groupby('family_id').size().rename('n_cases').to_frame())
display(
    standard_all_s4_accuracy.assign(
        accuracy=standard_all_s4_accuracy['accuracy'].map('{:.2%}'.format)
    )
)

In [10]:
%%script true
# Block 4 — Combine the three complete S4 cohorts and compare coarse accuracy.
full_s4_all_variant_results = pd.concat(
    [mild_all_s4_results, standard_all_s4_results, strong_all_s4_results],
    ignore_index=True,
)
full_s4_method_accuracy = pd.concat(
    [mild_all_s4_accuracy, standard_all_s4_accuracy, strong_all_s4_accuracy],
    ignore_index=True,
)
combined_method_rows = []
for method, (_, correct_column) in METHOD_SPECS.items():
    coarse_correct = f'coarse_{correct_column}'
    combined_method_rows.append({
        'variant_level': 'combined',
        'method': method,
        'n_cases': len(full_s4_all_variant_results),
        'n_correct': int(full_s4_all_variant_results[coarse_correct].sum()),
        'accuracy': float(full_s4_all_variant_results[coarse_correct].mean()),
    })
full_s4_method_accuracy = pd.concat(
    [full_s4_method_accuracy, pd.DataFrame(combined_method_rows)],
    ignore_index=True,
)
full_s4_family_rows = []
for method, (_, correct_column) in METHOD_SPECS.items():
    coarse_correct = f'coarse_{correct_column}'
    table = (
        full_s4_all_variant_results.groupby(
            ['variant_level', 'family_id', 'family_name'], as_index=False
        )
        .agg(
            n_cases=(coarse_correct, 'size'),
            n_correct=(coarse_correct, 'sum'),
            accuracy=(coarse_correct, 'mean'),
        )
    )
    table.insert(1, 'method', method)
    full_s4_family_rows.append(table)
    combined_table = (
        full_s4_all_variant_results.groupby(
            ['family_id', 'family_name'], as_index=False
        )
        .agg(
            n_cases=(coarse_correct, 'size'),
            n_correct=(coarse_correct, 'sum'),
            accuracy=(coarse_correct, 'mean'),
        )
    )
    combined_table.insert(0, 'variant_level', 'combined')
    combined_table.insert(1, 'method', method)
    full_s4_family_rows.append(combined_table)
full_s4_family_accuracy = pd.concat(full_s4_family_rows, ignore_index=True)
FULL_VARIANT_LEVELS = COMPARISON_VARIANTS + ['combined']
full_s4_method_accuracy['variant_level'] = pd.Categorical(
    full_s4_method_accuracy['variant_level'], FULL_VARIANT_LEVELS, ordered=True
)
full_s4_family_accuracy['variant_level'] = pd.Categorical(
    full_s4_family_accuracy['variant_level'], FULL_VARIANT_LEVELS, ordered=True
)

full_s4_fits_path = OUTPUT_DIR / 'full_s4_all_families_coarse_shape_fits.parquet'
full_s4_accuracy_path = OUTPUT_DIR / 'full_s4_all_families_coarse_method_accuracy.csv'
full_s4_family_path = OUTPUT_DIR / 'full_s4_all_families_coarse_family_accuracy.csv'
full_s4_all_variant_results.to_parquet(full_s4_fits_path, index=False)
full_s4_method_accuracy.to_csv(full_s4_accuracy_path, index=False)
full_s4_family_accuracy.to_csv(full_s4_family_path, index=False)

VARIANT_COLORS_FULL = {**VARIANT_COLORS, 'combined': '#CC79A7'}
fig, axes = plt.subplots(1, 2, figsize=(17.5, 5.8))
variant_positions = np.arange(len(FULL_VARIANT_LEVELS))
method_width = 0.19
for method_index, method in enumerate(method_order_variant):
    values = (
        full_s4_method_accuracy.loc[
            full_s4_method_accuracy['method'].eq(method)
        ].set_index('variant_level').reindex(FULL_VARIANT_LEVELS)['accuracy']
    )
    axes[0].bar(
        variant_positions + (method_index - 1.5) * method_width,
        values, width=method_width,
        color=METHOD_COLORS_VARIANT[method], label=method,
    )
axes[0].set_xticks(variant_positions)
axes[0].set_xticklabels([variant.title() for variant in FULL_VARIANT_LEVELS])
axes[0].set_ylim(0.0, 1.04)
axes[0].set_ylabel('Coarse shape-category accuracy')
axes[0].set_title('All S4-retained families')
axes[0].grid(True, axis='y', alpha=0.22)

primary_full_family = full_s4_family_accuracy.loc[
    full_s4_family_accuracy['method'].eq('b + derivatives (primary)')
]
all_family_ids = list(FULL_S4_COARSE_EXPECTED_SHAPE)
family_positions = np.arange(len(all_family_ids))
variant_width = 0.19
for variant_index, variant in enumerate(FULL_VARIANT_LEVELS):
    values = (
        primary_full_family.loc[
            primary_full_family['variant_level'].eq(variant)
        ].set_index('family_id').reindex(all_family_ids)['accuracy']
    )
    axes[1].bar(
        family_positions + (variant_index - 1.5) * variant_width,
        values, width=variant_width,
        color=VARIANT_COLORS_FULL[variant], label=variant.title(),
    )
axes[1].set_xticks(family_positions)
axes[1].set_xticklabels(all_family_ids)
axes[1].set_ylim(0.0, 1.04)
axes[1].set_ylabel('Primary-method coarse accuracy')
axes[1].set_title('By function family (missing bars = none passed S4)')
axes[1].grid(True, axis='y', alpha=0.22)
axes[1].legend(loc='lower right')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4, frameon=True)
fig.suptitle(
    'Full S4 cohorts — Coarse categories, not S-vs-Threshold identification',
    fontsize=15,
)
fig.tight_layout(rect=[0, 0.10, 1, 0.94])
full_s4_figure_path = FIGURE_DIR / 'full_s4_all_families_coarse_accuracy.png'
fig.savefig(full_s4_figure_path, dpi=FIGURE_DPI, bbox_inches='tight')
plt.close(fig)

print('All S4-retained families — combined coarse accuracy:')
display(
    full_s4_method_accuracy.assign(
        accuracy=full_s4_method_accuracy['accuracy'].map('{:.2%}'.format)
    ).sort_values(['variant_level', 'method'])
)
print(f'Saved: {full_s4_figure_path.relative_to(REPO_ROOT)}')
display(Image(filename=str(full_s4_figure_path)))

## Final label counts for every function family

These tables report the actual number of cases assigned to each primary derivative-method label. They expose `Other strong-monotonic`, `Uncertain`, and `Unclassified` directly rather than hiding them behind an accuracy percentage.

In [11]:
%%script true
PRIMARY_LABEL_ORDER = [
    'Linear', 'Concave', 'Convex',
    'Other strong-monotonic', 'Uncertain', 'Unclassified',
]
combined_label_source = full_s4_all_variant_results.copy()
combined_label_source['variant_level'] = 'combined'
label_count_source = pd.concat(
    [full_s4_all_variant_results, combined_label_source],
    ignore_index=True,
)
primary_label_counts = (
    label_count_source.groupby(
        ['variant_level', 'family_id', 'family_name', 'shape_b_derivative'],
        as_index=False,
    )
    .size()
    .rename(columns={
        'shape_b_derivative': 'predicted_shape',
        'size': 'n_cases',
    })
)
primary_label_counts['family_total'] = primary_label_counts.groupby(
    ['variant_level', 'family_id']
)['n_cases'].transform('sum')
primary_label_counts['share_within_family'] = (
    primary_label_counts['n_cases'] / primary_label_counts['family_total']
)
primary_label_counts_wide = (
    primary_label_counts.pivot_table(
        index=['variant_level', 'family_id', 'family_name'],
        columns='predicted_shape', values='n_cases',
        aggfunc='sum', fill_value=0,
    )
    .reindex(columns=PRIMARY_LABEL_ORDER, fill_value=0)
    .reset_index()
)
primary_label_counts_wide['Total'] = primary_label_counts_wide[
    PRIMARY_LABEL_ORDER
].sum(axis=1)
variant_order_map = {
    variant: order for order, variant in enumerate(FULL_VARIANT_LEVELS)
}
primary_label_counts_wide['_variant_order'] = (
    primary_label_counts_wide['variant_level'].map(variant_order_map)
)
primary_label_counts_wide = primary_label_counts_wide.sort_values(
    ['_variant_order', 'family_id']
).drop(columns='_variant_order').reset_index(drop=True)
primary_label_counts_wide = primary_label_counts_wide.rename(
    columns={'Other strong-monotonic': 'Other'}
)
PRIMARY_DISPLAY_COLUMNS = [
    'Linear', 'Concave', 'Convex', 'Other', 'Uncertain', 'Unclassified',
]
EXPECTED_DISPLAY_LABEL_BY_FAMILY = {
    family_id: (
        'Other' if expected == 'Other strong-monotonic' else expected
    )
    for family_id, expected in FULL_S4_COARSE_EXPECTED_SHAPE.items()
}
primary_label_counts_wide['expected_display_label'] = (
    primary_label_counts_wide['family_id'].map(EXPECTED_DISPLAY_LABEL_BY_FAMILY)
)
primary_label_counts_wide['n_correct'] = [
    int(row[row['expected_display_label']])
    for _, row in primary_label_counts_wide.iterrows()
]
primary_label_counts_wide['accuracy'] = (
    primary_label_counts_wide['n_correct'] / primary_label_counts_wide['Total']
)
primary_label_counts_formatted = primary_label_counts_wide.copy()
for label in PRIMARY_DISPLAY_COLUMNS:
    primary_label_counts_formatted[label] = [
        f'{int(count)} ({count / total:.1%})'
        for count, total in zip(
            primary_label_counts_wide[label],
            primary_label_counts_wide['Total'],
        )
    ]
primary_label_counts_formatted['Accuracy'] = [
    f'{int(correct)}/{int(total)} ({accuracy:.1%})'
    for correct, total, accuracy in zip(
        primary_label_counts_wide['n_correct'],
        primary_label_counts_wide['Total'],
        primary_label_counts_wide['accuracy'],
    )
]
primary_label_counts_formatted = primary_label_counts_formatted.loc[:, [
    'variant_level', 'family_id', 'family_name',
    *PRIMARY_DISPLAY_COLUMNS, 'Total', 'Accuracy',
]]

primary_label_counts_path = (
    OUTPUT_DIR / 'full_s4_all_families_primary_derivative_label_counts.csv'
)
primary_label_counts_formatted_path = (
    OUTPUT_DIR / 'full_s4_all_families_primary_derivative_label_counts_formatted.csv'
)
primary_label_counts_long_path = (
    OUTPUT_DIR / 'full_s4_all_families_primary_derivative_label_counts_long.csv'
)
primary_label_counts_wide.to_csv(primary_label_counts_path, index=False)
primary_label_counts_formatted.to_csv(
    primary_label_counts_formatted_path, index=False
)
primary_label_counts.to_csv(primary_label_counts_long_path, index=False)

TABLE_VARIANT_ORDER = ['strong', 'mild', 'standard', 'combined']
variant_label_tables = {}
variant_label_table_paths = {}
for variant in TABLE_VARIANT_ORDER:
    variant_table = primary_label_counts_formatted.loc[
        primary_label_counts_formatted['variant_level'].eq(variant),
        ['family_id', 'family_name', *PRIMARY_DISPLAY_COLUMNS, 'Total', 'Accuracy'],
    ].reset_index(drop=True)
    variant_label_tables[variant] = variant_table
    variant_path = OUTPUT_DIR / (
        f'{variant}_primary_derivative_label_table_formatted.csv'
    )
    variant_table.to_csv(variant_path, index=False)
    variant_label_table_paths[variant] = variant_path
    print(f'{variant.title()} — final primary labels by true family:')
    print(variant_table.to_string(index=False))
    print()
print(f'Saved numeric table:   {primary_label_counts_path.relative_to(REPO_ROOT)}')
print(f'Saved formatted table: {primary_label_counts_formatted_path.relative_to(REPO_ROOT)}')
print('Saved four separate tables:')
for variant in TABLE_VARIANT_ORDER:
    print(f'  {variant.title()}: {variant_label_table_paths[variant].relative_to(REPO_ROOT)}')

## Save pilot results and family/variant summaries

In [12]:
%%script true
variant_run_tag = '_'.join(enabled_variants)
scope_run_tag = 'noise_free_pilot' if NOISE_FREE_PILOT_ONLY else 'all_noise_levels'
run_label = f'{variant_run_tag}_only_{scope_run_tag}'
full_path = OUTPUT_DIR / f'power_law_{run_label}_fits_full.parquet'
manifest_path = OUTPUT_DIR / f'power_law_{run_label}_manifest.csv'
summary_path = OUTPUT_DIR / f'power_law_{run_label}_summary_by_family_variant.csv'
failures_path = OUTPUT_DIR / f'power_law_{run_label}_fit_failures.csv'
config_path = OUTPUT_DIR / f'power_law_{run_label}_config.json'
accuracy_by_noise_path = OUTPUT_DIR / f'power_law_{run_label}_accuracy_by_noise.csv'
family_accuracy_by_noise_path = OUTPUT_DIR / f'power_law_{run_label}_family_accuracy_by_noise.csv'
cumulative_accuracy_path = OUTPUT_DIR / f'power_law_{run_label}_cumulative_accuracy.csv'
method_accuracy_path = OUTPUT_DIR / f'power_law_{run_label}_method_accuracy.csv'
method_family_accuracy_path = OUTPUT_DIR / f'power_law_{run_label}_method_family_accuracy.csv'
method_accuracy_by_noise_path = OUTPUT_DIR / f'power_law_{run_label}_method_accuracy_by_noise.csv'
method_cumulative_accuracy_path = OUTPUT_DIR / f'power_law_{run_label}_method_cumulative_accuracy.csv'

output_paths = [
    full_path, manifest_path, summary_path, failures_path, config_path,
    accuracy_by_noise_path, family_accuracy_by_noise_path,
    cumulative_accuracy_path, method_accuracy_path,
    method_family_accuracy_path, method_accuracy_by_noise_path,
    method_cumulative_accuracy_path,
]
for path in output_paths:
    if path.exists() and not OVERWRITE:
        raise FileExistsError(path)

fit_results.to_parquet(full_path, index=False)
manifest_columns = [
    'candidate_index', 'case_id', 'family_id', 'family_name',
    'variant_level', 'repeat', 'inverse_snr', 'MIC',
    'pearson_r', 'spearman_rho', 'fit_converged', 'fit_acceptable',
    'a', 'b', 'c', 'b_se', 'b_ci_lower', 'b_ci_upper',
    'abs_b_minus_one', 'bic_evidence',
    'r2', 'cv_r2', 'nrmse', 'power_bic', 'linear_bic',
    'delta_bic_linear_minus_power',
    'f1_q_positive', 'f1_q_negative', 'f1_dominance', 'f1_dominant_sign',
    'f2_q_positive', 'f2_q_negative', 'f2_dominance', 'f2_dominant_sign',
    'curvature_strength', 'runs_count', 'runs_z', 'runs_p_lower',
    'residual_relative_rmse_range', 'runs_structured_residual',
    'shape_b_only', 'shape_b_derivative', 'shape_b_runs', 'shape_b_r2',
    'power_law_shape', 'fit_error', 'expected_power_law_shape',
    'shape_evaluable', 'correct_b_only', 'correct_b_derivative',
    'correct_b_runs', 'correct_b_r2', 'shape_correct',
]
fit_results.loc[:, manifest_columns].to_csv(manifest_path, index=False)

shape_counts = (
    fit_results.groupby(
        ['family_id', 'family_name', 'variant_level', 'power_law_shape'],
        as_index=False,
    )
    .agg(
        n_cases=('case_id', 'size'),
        median_b=('b', 'median'),
        median_cv_r2=('cv_r2', 'median'),
        median_nrmse=('nrmse', 'median'),
        median_delta_bic=('delta_bic_linear_minus_power', 'median'),
    )
)
shape_counts['share_within_family_variant'] = (
    shape_counts['n_cases']
    / shape_counts.groupby(['family_id', 'variant_level'])['n_cases'].transform('sum')
)
shape_counts.to_csv(summary_path, index=False)
fit_results.loc[~fit_results['fit_converged'], manifest_columns].to_csv(
    failures_path, index=False
)
accuracy_by_noise_level.to_csv(accuracy_by_noise_path, index=False)
family_accuracy_by_noise_level.to_csv(family_accuracy_by_noise_path, index=False)
cumulative_accuracy_by_noise.to_csv(cumulative_accuracy_path, index=False)
method_accuracy_summary.to_csv(method_accuracy_path, index=False)
method_family_accuracy.to_csv(method_family_accuracy_path, index=False)
method_accuracy_by_noise.to_csv(method_accuracy_by_noise_path, index=False)
method_cumulative_accuracy.to_csv(method_cumulative_accuracy_path, index=False)

config = {
    'S3_DATA_DIR': S3_DATA_DIR,
    'S4_DATA_DIR': S4_DATA_DIR,
    'scatter_points_path': str(SCATTER_POINTS_PATH),
    'strong_monotonicity_path': str(STRONG_MONOTONICITY_PATH),
    'output_dir': str(OUTPUT_DIR),
    'noise_free_pilot_only': NOISE_FREE_PILOT_ONLY,
    'variant_switches': variant_switches,
    'enabled_variants': enabled_variants,
    'n_parent_cases': len(strong_monotonicity_cases),
    'n_analysis_cases': len(analysis_cases),
    'power_exponent_bounds': [POWER_EXPONENT_MIN, POWER_EXPONENT_MAX],
    'cv_folds': CV_FOLDS,
    'primary_method': 'b + derivatives',
    'first_derivative_dominance_threshold': FIRST_DERIVATIVE_DOMINANCE_THRESHOLD,
    'second_derivative_dominance_threshold': SECOND_DERIVATIVE_DOMINANCE_THRESHOLD,
    'derivative_interior_margin': DERIVATIVE_INTERIOR_MARGIN,
    'derivative_grid_size': DERIVATIVE_GRID_SIZE,
    'runs_z_threshold': RUNS_Z_THRESHOLD,
    'runs_numerical_relative_rmse_tolerance': RUNS_NUMERICAL_RELATIVE_RMSE_TOLERANCE,
    'r2_comparison_threshold': R2_COMPARISON_THRESHOLD,
    'diagnostic_min_cv_r2': MIN_ACCEPTABLE_CV_R2,
    'diagnostic_max_nrmse': MAX_ACCEPTABLE_NRMSE,
    'delta_bic_weak_threshold': DELTA_BIC_WEAK_THRESHOLD,
    'delta_bic_strong_threshold': DELTA_BIC_STRONG_THRESHOLD,
    'power_exponent_linear_tolerance': POWER_EXPONENT_LINEAR_TOLERANCE,
    'expected_shape_by_family': EXPECTED_SHAPE_BY_FAMILY,
    'overall_known_family_accuracy': float(evaluated_results['shape_correct'].mean()),
    'method_accuracies': {
        row['method']: float(row['accuracy'])
        for _, row in method_accuracy_summary.iterrows()
    },
    'b_confidence_level': B_CONFIDENCE_LEVEL,
    'random_seed': RANDOM_SEED,
}
config_path.write_text(json.dumps(config, indent=2), encoding='utf-8')

print('Saved:')
for path in output_paths:
    print(f'  {path.relative_to(REPO_ROOT)}')
display(shape_counts)

## Diagnostic figures

In [13]:
%%script true
FAMILY_ORDER = ['F01', 'F03', 'F05', 'F07', 'F13', 'F15', 'F23']
FAMILY_COLORS = {
    'F01': '#002B5B', 'F03': '#0065BD', 'F05': '#42B7E9',
    'F07': '#F2C14E', 'F13': '#00B8B0', 'F15': '#007F5F',
    'F23': '#FF5A36',
}
FAMILY_LABELS = {
    'F01': 'F01 Linear', 'F03': 'F03 Power convex',
    'F05': 'F05 Power concave', 'F07': 'F07 Saturation',
    'F13': 'F13 S-curve', 'F15': 'F15 Threshold',
    'F23': 'F23 Two Lines',
}
family_order = [fid for fid in FAMILY_ORDER if fid in set(fit_results['family_id'])]
scope_title = 'Noise-free' if NOISE_FREE_PILOT_ONLY else 'All retained noise levels'
figure_paths = []


def draw_family_violin(metric: str, ylabel: str, filename: str, reference=None):
    distributions = [
        fit_results.loc[
            fit_results['family_id'].eq(fid) & fit_results['fit_converged'], metric
        ].dropna().to_numpy()
        for fid in family_order
    ]
    positions = np.arange(len(family_order))
    fig, ax = plt.subplots(figsize=(9.0, 5.6))
    violins = ax.violinplot(
        distributions, positions=positions, showmedians=True,
        showextrema=False, widths=0.80,
    )
    for fid, body in zip(family_order, violins['bodies']):
        body.set_facecolor(FAMILY_COLORS[fid])
        body.set_edgecolor(FAMILY_COLORS[fid])
        body.set_alpha(0.55)
    violins['cmedians'].set_color('#222222')
    if reference is not None:
        ax.axhline(reference, color='#333333', linestyle='--', linewidth=1.0)
    ax.set_xticks(positions)
    ax.set_xticklabels([FAMILY_LABELS[fid] for fid in family_order], rotation=25, ha='right')
    ax.set_ylabel(ylabel)
    ax.set_title(f'{scope_title}, Strong variant only — {ylabel}')
    ax.grid(True, axis='y', color='#B0B0B0', alpha=0.25)
    fig.tight_layout()
    path = FIGURE_DIR / filename
    fig.savefig(path, dpi=FIGURE_DPI, bbox_inches='tight')
    plt.close(fig)
    return path


figure_paths.append(draw_family_violin('b', 'Power exponent b', f'{run_label}_power_exponent_by_family.png', reference=1.0))
figure_paths.append(draw_family_violin('cv_r2', 'Cross-validated R²', f'{run_label}_cv_r2_by_family.png', reference=MIN_ACCEPTABLE_CV_R2))

print('Saved distribution figures:')
for path in figure_paths:
    print(f'  {path.relative_to(REPO_ROOT)}')
    display(Image(filename=str(path)))


def configure_inverse_snr_axis(ax):
    ax.set_xscale('symlog', linthresh=1e-4, linscale=1.0)
    ticks = [0.0, 1e-4, 1e-3, 1e-2, 1e-1]
    ax.set_xticks(ticks)
    ax.set_xticklabels(['No noise', '10⁻⁴', '10⁻³', '10⁻²', '10⁻¹'])
    ax.set_xlim(0.0, float(accuracy_by_noise_level['inverse_snr'].max()) * 1.05)
    ax.set_xlabel('1 / SNR (more noise →)')


METHOD_COLORS = {
    'b only': '#7A7A7A',
    'b + derivatives (primary)': '#009E73',
    'b + Runs test': '#D55E00',
    'b + R²': '#7B61A8',
}
method_order = list(METHOD_SPECS)
fig, axes = plt.subplots(1, 3, figsize=(18.0, 5.6))

# Overall accuracy.
overall_plot = method_accuracy_summary.set_index('method').loc[method_order]
bars = axes[0].bar(
    np.arange(len(method_order)), overall_plot['accuracy'],
    color=[METHOD_COLORS[method] for method in method_order],
)
axes[0].set_xticks(np.arange(len(method_order)))
axes[0].set_xticklabels(method_order, rotation=24, ha='right')
axes[0].set_ylim(0.65, 1.02)
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Overall')
axes[0].grid(True, axis='y', alpha=0.22)
for bar, accuracy in zip(bars, overall_plot['accuracy']):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2, accuracy + 0.008,
        f'{accuracy:.1%}', ha='center', va='bottom', fontsize=9,
    )

# Accuracy within each known family.
family_ids = [fid for fid in family_order if fid in set(evaluated_results['family_id'])]
x_positions = np.arange(len(family_ids))
bar_width = 0.19
for method_index, method in enumerate(method_order):
    family_values = (
        method_family_accuracy.loc[
            method_family_accuracy['method'].eq(method)
        ].set_index('family_id').reindex(family_ids)['accuracy']
    )
    axes[1].bar(
        x_positions + (method_index - 1.5) * bar_width,
        family_values,
        width=bar_width,
        color=METHOD_COLORS[method],
        label=method,
    )
axes[1].set_xticks(x_positions)
axes[1].set_xticklabels([FAMILY_LABELS[fid] for fid in family_ids], rotation=18, ha='right')
axes[1].set_ylim(0.0, 1.04)
axes[1].set_ylabel('Accuracy')
axes[1].set_title('By function family')
axes[1].grid(True, axis='y', alpha=0.22)

# Cumulative accuracy as progressively noisier retained cases are added.
for method in method_order:
    curve = method_cumulative_accuracy.loc[
        method_cumulative_accuracy['method'].eq(method)
    ]
    axes[2].plot(
        curve['max_inverse_snr'], curve['cumulative_accuracy'],
        color=METHOD_COLORS[method], linewidth=2.0, label=method,
    )
axes[2].set_ylim(0.65, 1.02)
axes[2].set_ylabel('Cumulative accuracy')
axes[2].set_title('As noise is added')
axes[2].grid(True, alpha=0.22)
configure_inverse_snr_axis(axes[2])

handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4, frameon=True)
fig.suptitle('Strong variant — Four shape-classification rules', fontsize=15)
fig.tight_layout(rect=[0, 0.10, 1, 0.94])
method_comparison_path = FIGURE_DIR / f'{run_label}_method_accuracy_comparison.png'
fig.savefig(method_comparison_path, dpi=FIGURE_DPI, bbox_inches='tight')
plt.close(fig)
figure_paths.append(method_comparison_path)
display(Image(filename=str(method_comparison_path)))


if not NOISE_FREE_PILOT_ONLY:
    fig, axes = plt.subplots(2, 1, figsize=(10.0, 8.0), sharex=True)
    axes[0].plot(
        accuracy_by_noise_level['inverse_snr'],
        accuracy_by_noise_level['accuracy'],
        color='#0065BD', linewidth=1.8, label='Accuracy at each noise level',
    )
    axes[0].plot(
        cumulative_accuracy_by_noise['max_inverse_snr'],
        cumulative_accuracy_by_noise['cumulative_accuracy'],
        color='#D55E00', linewidth=1.8, label='Cumulative accuracy',
    )
    axes[0].set_ylim(-0.02, 1.02)
    axes[0].set_ylabel('Known-family accuracy')
    axes[0].grid(True, alpha=0.25)
    axes[0].legend(loc='lower left')

    axes[1].plot(
        accuracy_by_noise_level['inverse_snr'],
        accuracy_by_noise_level['n_cases'],
        color='#333333', linewidth=1.8,
    )
    axes[1].set_ylabel('S4-retained cases')
    axes[1].grid(True, alpha=0.25)
    configure_inverse_snr_axis(axes[1])
    fig.suptitle('Strong variant — Shape accuracy as noise is added', fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    accuracy_figure_path = FIGURE_DIR / f'{run_label}_accuracy_vs_noise.png'
    fig.savefig(accuracy_figure_path, dpi=FIGURE_DPI, bbox_inches='tight')
    plt.close(fig)
    figure_paths.append(accuracy_figure_path)
    display(Image(filename=str(accuracy_figure_path)))

    fig, ax = plt.subplots(figsize=(10.0, 5.8))
    for family_id in family_order:
        family_curve = family_accuracy_by_noise_level.loc[
            family_accuracy_by_noise_level['family_id'].eq(family_id)
        ]
        ax.plot(
            family_curve['inverse_snr'], family_curve['accuracy'],
            color=FAMILY_COLORS[family_id], linewidth=1.8,
            label=FAMILY_LABELS[family_id],
        )
    ax.set_ylim(-0.02, 1.02)
    ax.set_ylabel('Accuracy at each noise level')
    configure_inverse_snr_axis(ax)
    ax.grid(True, alpha=0.25)
    ax.legend(loc='lower left', ncol=2)
    ax.set_title('Strong variant — Family-specific shape accuracy')
    fig.tight_layout()
    family_accuracy_figure_path = FIGURE_DIR / f'{run_label}_family_accuracy_vs_noise.png'
    fig.savefig(family_accuracy_figure_path, dpi=FIGURE_DPI, bbox_inches='tight')
    plt.close(fig)
    figure_paths.append(family_accuracy_figure_path)
    display(Image(filename=str(family_accuracy_figure_path)))

In [14]:
%%script true
representatives = []
for family_id in family_order:
    candidates = fit_results.loc[
        fit_results['family_id'].eq(family_id) & fit_results['fit_converged']
    ].copy()
    if candidates.empty:
        continue
    median_score = candidates['cv_r2'].median()
    representative_index = (candidates['cv_r2'] - median_score).abs().idxmin()
    representatives.append(fit_results.loc[representative_index])

n_columns = 3
n_rows = int(np.ceil(len(representatives) / n_columns))
fig, axes = plt.subplots(n_rows, n_columns, figsize=(12.0, 3.6 * n_rows))
axes = np.atleast_1d(axes).ravel()
for ax, case in zip(axes, representatives):
    row = int(case['pilot_row'])
    x = analysis_x[row]
    y = analysis_y[row]
    order = np.argsort(x)
    ax.scatter(x, y, s=7, alpha=0.28, color=FAMILY_COLORS[case['family_id']], edgecolors='none')
    ax.plot(
        x[order], power_law(x[order], case['a'], case['b'], case['c']),
        color='#111111', linewidth=1.5,
    )
    display_shape = case['power_law_shape'].replace('Other strong-monotonic', 'Other')
    ax.set_title(
        f"{FAMILY_LABELS[case['family_id']]} — {case['variant_level']}\n"
        f"b={case['b']:.3f}, D₂={case['f2_dominance']:.3f}, {display_shape}",
        fontsize=9.5,
    )
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.grid(True, alpha=0.18)
for ax in axes[len(representatives):]:
    ax.axis('off')
fig.suptitle(f'Representative {scope_title.lower()} Strong-variant power-law fits', y=0.995, fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.97])
representative_path = FIGURE_DIR / f'{run_label}_representative_power_law_fits.png'
fig.savefig(representative_path, dpi=FIGURE_DPI, bbox_inches='tight')
plt.close(fig)
figure_paths.append(representative_path)

print(f'Saved: {representative_path.relative_to(REPO_ROOT)}')
display(Image(filename=str(representative_path)))

# Simplified hierarchical S5 — MIC ≥ 0.8 full cohort

This section implements the primary sequential tree without Runs-test, R²,
NRMSE, spline, LOWESS, or GAM routing:

Simple/Complex → Strong monotonicity → free Power-law confirmation of Linear
(the Pearson strong-linearity gate is disabled) → one Cubic polynomial fit for every
remaining Simple, non-linear case.

The Cubic derivative rule is evaluated on the central 70% of the observed
\(x\)-range. It uses strict first-derivative sign consistency and
\(D_{f''}=0.75\) to report Concave, Convex, S-shaped, or Other/Uncertain.
R² and NRMSE are diagnostics only.
Each step keeps its switches and thresholds at the beginning of that step.

In [15]:
# ============================================================
# STEP 1 CONFIG — INPUT / OUTPUT
# ============================================================
SIMPLE_TREE_MIC_INPUT_FILENAME = 'mic_0p8_to_1_full.parquet'
SIMPLE_TREE_OUTPUT_SUBDIR = 'simple_hierarchy_mic_ge_0p8'
SIMPLE_TREE_OVERWRITE = True

SIMPLE_TREE_MIC_INPUT_PATH = S4_DIR / SIMPLE_TREE_MIC_INPUT_FILENAME
SIMPLE_TREE_OUTPUT_DIR = OUTPUT_DIR / SIMPLE_TREE_OUTPUT_SUBDIR
SIMPLE_TREE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not SIMPLE_TREE_MIC_INPUT_PATH.exists():
    raise FileNotFoundError(SIMPLE_TREE_MIC_INPUT_PATH)

points = np.load(SCATTER_POINTS_PATH)
if set(points.files) != {'x', 'y'}:
    raise KeyError(f'Expected x and y arrays, found {points.files}.')
if points['x'].shape != points['y'].shape:
    raise ValueError('x and y arrays have different shapes.')

simple_tree_cases = pd.read_parquet(SIMPLE_TREE_MIC_INPUT_PATH)
simple_tree_required_columns = {
    'candidate_index', 'case_id', 'family_id', 'family_name',
    'variant_level', 'repeat', 'inverse_snr', 'MIC',
    'pearson_r', 'spearman_rho',
}
simple_tree_missing = simple_tree_required_columns.difference(
    simple_tree_cases.columns
)
if simple_tree_missing:
    raise KeyError(f'Missing MIC≥0.8 columns: {sorted(simple_tree_missing)}')
if simple_tree_cases['candidate_index'].duplicated().any():
    raise ValueError('MIC≥0.8 input contains duplicate candidate_index values.')

simple_tree_cases = simple_tree_cases.sort_values(
    'candidate_index'
).reset_index(drop=True)
simple_tree_indices = simple_tree_cases['candidate_index'].to_numpy(dtype=int)
simple_tree_x = points['x'][simple_tree_indices].astype(float)
simple_tree_y = points['y'][simple_tree_indices].astype(float)
print(f'MIC≥0.8 input: {len(simple_tree_cases):,} cases')
print(simple_tree_cases.groupby(['variant_level', 'family_id']).size().to_string())

Future exception was never retrieved
future: <Future finished exception=BrokenPipeError(32, 'Broken pipe')>
Traceback (most recent call last):
  File "/Users/mimi/miniconda3/envs/pip39/lib/python3.9/asyncio/unix_events.py", line 687, in _write_ready
    n = os.write(self._fileno, self._buffer)
BrokenPipeError: [Errno 32] Broken pipe


Future exception was never retrieved
future: <Future finished exception=BrokenPipeError(32, 'Broken pipe')>
Traceback (most recent call last):
  File "/Users/mimi/miniconda3/envs/pip39/lib/python3.9/asyncio/unix_events.py", line 687, in _write_ready
    n = os.write(self._fileno, self._buffer)
BrokenPipeError: [Errno 32] Broken pipe


MIC≥0.8 input: 24,643 cases
variant_level  family_id
mild           F01           963
               F03           956
               F05           955
               F07           895
               F13          1059
               F15          1086
               F18           944
               F21           531
               F22          1047
               F23           200
               F25            32
               F26            38
standard       F01           988
               F03           873
               F05           926
               F07           774
               F13          1084
               F15          1110
               F18           932
               F21           519
               F22          1003
               F23           239
               F25            22
               F26            10
strong         F01           994
               F03           806
               F05           856
               F07            55
               F13     

In [16]:
# ============================================================
# STEP 2 CONFIG — SIMPLE / COMPLEX
# ============================================================
SIMPLE_TREE_USE_SIMPLE_COMPLEX_GATE = True
SIMPLE_TREE_MIN_ABS_CORRELATION_THRESHOLD = 0.70

if SIMPLE_TREE_USE_SIMPLE_COMPLEX_GATE:
    simple_tree_cases['simple_relationship'] = (
        simple_tree_cases['pearson_r'].abs().ge(
            SIMPLE_TREE_MIN_ABS_CORRELATION_THRESHOLD
        )
        & simple_tree_cases['spearman_rho'].abs().ge(
            SIMPLE_TREE_MIN_ABS_CORRELATION_THRESHOLD
        )
    )
else:
    simple_tree_cases['simple_relationship'] = True
simple_tree_cases['simple_complex_label'] = np.where(
    simple_tree_cases['simple_relationship'], 'Simple', 'Complex'
)

# ============================================================
# STEP 3 CONFIG — STRONG MONOTONICITY WITHIN SIMPLE
# ============================================================
SIMPLE_TREE_USE_STRONG_MONOTONICITY_GATE = True
SIMPLE_TREE_STRONG_MONOTONIC_SPEARMAN_THRESHOLD = 0.90

if SIMPLE_TREE_USE_STRONG_MONOTONICITY_GATE:
    simple_tree_cases['strong_monotonic'] = (
        simple_tree_cases['simple_relationship']
        & simple_tree_cases['spearman_rho'].abs().ge(
            SIMPLE_TREE_STRONG_MONOTONIC_SPEARMAN_THRESHOLD
        )
    )
else:
    simple_tree_cases['strong_monotonic'] = (
        simple_tree_cases['simple_relationship']
    )
simple_tree_cases['simple_not_strong_monotonic'] = (
    simple_tree_cases['simple_relationship']
    & ~simple_tree_cases['strong_monotonic']
)

# ============================================================
# STEP 4 CONFIG — STRONG LINEARITY CANDIDATE
# ============================================================
SIMPLE_TREE_USE_STRONG_LINEARITY_GATE = False
SIMPLE_TREE_STRONG_LINEARITY_PEARSON_THRESHOLD = 0.90

if SIMPLE_TREE_USE_STRONG_LINEARITY_GATE:
    simple_tree_cases['strong_linearity_candidate'] = (
        simple_tree_cases['strong_monotonic']
        & simple_tree_cases['pearson_r'].abs().gt(
            SIMPLE_TREE_STRONG_LINEARITY_PEARSON_THRESHOLD
        )
    )
else:
    simple_tree_cases['strong_linearity_candidate'] = (
        simple_tree_cases['strong_monotonic']
    )
simple_tree_cases['strong_monotonic_not_strong_linearity'] = (
    simple_tree_cases['strong_monotonic']
    & ~simple_tree_cases['strong_linearity_candidate']
)

print('Simple:', f"{simple_tree_cases['simple_relationship'].sum():,}")
print('Complex:', f"{(~simple_tree_cases['simple_relationship']).sum():,}")
print('Strong monotonic within Simple:', f"{simple_tree_cases['strong_monotonic'].sum():,}")
print(
    'Strong-linearity candidates:',
    f"{simple_tree_cases['strong_linearity_candidate'].sum():,}",
)

Simple: 18,669
Complex: 5,974
Strong monotonic within Simple: 12,908
Strong-linearity candidates: 12,908


In [17]:
# ============================================================
# STEP 5 CONFIG — FREE POWER-LAW LINEAR CONFIRMATION
# ============================================================
SIMPLE_TREE_POWER_EXPONENT_MIN = 0.05
SIMPLE_TREE_POWER_EXPONENT_MAX = 8.0
SIMPLE_TREE_POWER_EXPONENT_LINEAR_TOLERANCE = 0.05
# Model: y = a*x**b + c. Linear is confirmed when |b - 1| <= tolerance.


def simple_tree_power_law(
    x: np.ndarray, a: float, b: float, c: float
) -> np.ndarray:
    return c + a * np.power(x, b)


def simple_tree_initial_power_parameters(
    x: np.ndarray, y: np.ndarray
) -> tuple[float, float, float]:
    y_low = float(np.quantile(y, 0.05))
    y_high = float(np.quantile(y, 0.95))
    amplitude = max(y_high - y_low, np.finfo(float).eps)
    covariance = float(np.cov(x, y, ddof=1)[0, 1])
    direction = -1.0 if covariance < 0 else 1.0
    a0 = direction * amplitude
    c0 = y_low if direction > 0 else y_high
    return a0, 1.0, c0


def simple_tree_normalized_rmse(
    y_true: np.ndarray, y_pred: np.ndarray
) -> float:
    scale = float(np.std(y_true, ddof=1))
    if not np.isfinite(scale) or scale <= np.finfo(float).eps:
        return np.nan
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)) / scale)


def simple_tree_information_criterion(
    y_true: np.ndarray, y_pred: np.ndarray, parameter_count: int
) -> float:
    n = len(y_true)
    rss = max(
        float(np.sum((y_true - y_pred) ** 2)),
        np.finfo(float).tiny,
    )
    return float(n * np.log(rss / n) + parameter_count * np.log(n))


def simple_tree_fit_power_parameters(x: np.ndarray, y: np.ndarray):
    p0 = simple_tree_initial_power_parameters(x, y)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', OptimizeWarning)
        return curve_fit(
            simple_tree_power_law, x, y, p0=p0,
            bounds=(
                [-np.inf, SIMPLE_TREE_POWER_EXPONENT_MIN, -np.inf],
                [np.inf, SIMPLE_TREE_POWER_EXPONENT_MAX, np.inf],
            ),
            maxfev=30000,
        )


def simple_tree_power_features(x: np.ndarray, y: np.ndarray) -> dict:
    result = {
        'fit_converged': False, 'fit_error': '',
        'a': np.nan, 'b': np.nan, 'c': np.nan,
        'r2_diagnostic': np.nan, 'nrmse_diagnostic': np.nan,
        'bic': np.nan,
    }
    try:
        parameters, _ = simple_tree_fit_power_parameters(x, y)
        a, b, c = map(float, parameters)
        prediction = simple_tree_power_law(x, a, b, c)
        result.update({
            'fit_converged': True,
            'a': a, 'b': b, 'c': c,
            'r2_diagnostic': float(r2_score(y, prediction)),
            'nrmse_diagnostic': simple_tree_normalized_rmse(y, prediction),
            'bic': simple_tree_information_criterion(
                y, prediction, parameter_count=3
            ),
        })
    except Exception as error:
        result['fit_error'] = f'{type(error).__name__}: {error}'
    return result


def simple_tree_attach_features(
    frame, row_indices, feature_rows, prefix, not_eligible_message
):
    feature_frame = pd.DataFrame(feature_rows, index=row_indices).add_prefix(prefix)
    for column in feature_frame.columns:
        if pd.api.types.is_bool_dtype(feature_frame[column]):
            frame[column] = False
        elif column.endswith('error'):
            frame[column] = not_eligible_message
        else:
            frame[column] = np.nan
        frame.loc[row_indices, column] = feature_frame[column].to_numpy()
    return frame


simple_tree_results = simple_tree_cases.copy()

# Pearson gate OFF: every Strong-monotonic case receives the Linear power-law fit.
linear_fit_indices = np.flatnonzero(
    simple_tree_results['strong_linearity_candidate'].to_numpy()
)
print(f'Linear power-law eligible: {len(linear_fit_indices):,}')
linear_fit_rows = [
    simple_tree_power_features(simple_tree_x[index], simple_tree_y[index])
    for index in linear_fit_indices
]
simple_tree_results = simple_tree_attach_features(
    simple_tree_results, linear_fit_indices, linear_fit_rows, 'linear_',
    'Not eligible: did not pass Strong monotonicity',
)
simple_tree_results['linear_abs_b_minus_one'] = (
    simple_tree_results['linear_b'] - 1.0
).abs()
simple_tree_results['confirmed_linear'] = (
    simple_tree_results['strong_linearity_candidate']
    & simple_tree_results['linear_fit_converged']
    & simple_tree_results['linear_abs_b_minus_one'].le(
        SIMPLE_TREE_POWER_EXPONENT_LINEAR_TOLERANCE
    )
)
simple_tree_results['strong_linearity_not_linear'] = (
    simple_tree_results['strong_linearity_candidate']
    & ~simple_tree_results['confirmed_linear']
)

# The three rejected Simple branches are disjoint and are merged only after
# their individual counts have been retained.
shape_branch_1 = simple_tree_results['simple_not_strong_monotonic']
shape_branch_2 = simple_tree_results[
    'strong_monotonic_not_strong_linearity'
]
shape_branch_3 = simple_tree_results['strong_linearity_not_linear']
branch_membership_count = (
    shape_branch_1.astype(int)
    + shape_branch_2.astype(int)
    + shape_branch_3.astype(int)
)
if branch_membership_count.gt(1).any():
    raise ValueError('Shape-pool routing branches overlap.')
simple_tree_results['shape_branch_1'] = shape_branch_1
simple_tree_results['shape_branch_2'] = shape_branch_2
simple_tree_results['shape_branch_3'] = shape_branch_3
simple_tree_results['remaining_shape_pool'] = (
    shape_branch_1 | shape_branch_2 | shape_branch_3
)

print('Sequential routing counts before curvature:')
print(pd.Series({
    'Complex excluded before fitting': int(
        (~simple_tree_results['simple_relationship']).sum()
    ),
    'Linear power-law eligible': len(linear_fit_indices),
    'Confirmed Linear': int(simple_tree_results['confirmed_linear'].sum()),
    'Shape branch 1: Simple not Strong monotonic': int(shape_branch_1.sum()),
    'Shape branch 2: Strong monotonic not Strong linearity': int(
        shape_branch_2.sum()
    ),
    'Shape branch 3: Strong linearity not Linear': int(shape_branch_3.sum()),
    'Merged cubic curvature pool': int(
        simple_tree_results['remaining_shape_pool'].sum()
    ),
}).to_string())

Linear power-law eligible: 12,908


Sequential routing counts before curvature:
Complex excluded before fitting                           5974
Linear power-law eligible                                12908
Confirmed Linear                                          2877
Shape branch 1: Simple not Strong monotonic               5761
Shape branch 2: Strong monotonic not Strong linearity        0
Shape branch 3: Strong linearity not Linear              10031
Merged cubic curvature pool                              15792


## Step 6 — Cubic-only curvature classification

Linear cases have already been removed by the preceding free Power-law step.
Every remaining Simple, non-linear case is fitted with one cubic polynomial:

\[
f(x)=a_3x^3+a_2x^2+a_1x+a_0.
\]

There is no Power-law/Cubic BIC comparison in this step. Evaluate the
derivatives on the central 70% of the observed \(x\)-range:

\[
f'(x)=3a_3x^2+2a_2x+a_1,\qquad
f''(x)=6a_3x+2a_2.
\]

Curvature direction is weighted by the absolute magnitude of \(f''(x)\):

\[
q_+=
\frac{\sum_{f''(x_i)>0}|f''(x_i)|}
{\sum_i|f''(x_i)|},
\qquad
q_-=
\frac{\sum_{f''(x_i)<0}|f''(x_i)|}
{\sum_i|f''(x_i)|},
\]

\[
D_{f''}=\max(q_+,q_-).
\]

The first-derivative condition is strict: \(f'(x)\) must not change sign
anywhere on the central grid. No 90% direction-consistency relaxation is used.

| Condition | Result |
|---|---|
| \(f'(x)\) changes sign | Other/Uncertain |
| \(f'(x)\) does not change sign, \(D_{f''}\ge0.75\), dominant \(f''>0\) | Convex |
| \(f'(x)\) does not change sign, \(D_{f''}\ge0.75\), dominant \(f''<0\) | Concave |
| \(f'(x)\) does not change sign, \(D_{f''}<0.75\), and \(f''(x)\) changes sign once | S-shaped |
| Any other case | Other/Uncertain |

\(R^2\) is retained as a diagnostic only; it is not a classification gate.

In [18]:
# ============================================================
# STEP 6 CONFIG — CUBIC-ONLY CURVATURE CLASSIFICATION
# ============================================================
SIMPLE_TREE_CUBIC_INTERIOR_MARGIN = 0.15
SIMPLE_TREE_CUBIC_GRID_SIZE = 200
SIMPLE_TREE_CURVATURE_DOMINANCE_THRESHOLD = 0.75
# Every remaining Simple, non-linear case receives one Cubic fit. Derivatives
# are evaluated on the central 70%, with a strict no-sign-change rule for f'.


def simple_tree_count_sign_changes(values: np.ndarray) -> int:
    signs = np.sign(np.asarray(values, dtype=float)).copy()
    for index in range(1, len(signs)):
        if signs[index] == 0:
            signs[index] = signs[index - 1]
    return int(np.sum(
        (signs[1:] != signs[:-1])
        & (signs[1:] != 0)
        & (signs[:-1] != 0)
    ))


def simple_tree_signed_absolute_dominance(values: np.ndarray) -> dict:
    values = np.asarray(values, dtype=float)
    absolute_values = np.abs(values)
    total_mass = float(absolute_values.sum())
    if total_mass <= np.finfo(float).eps:
        return {
            'q_positive': 0.5, 'q_negative': 0.5,
            'dominance': 1.0, 'dominant_sign': 0,
        }
    q_positive = float(absolute_values[values > 0].sum() / total_mass)
    q_negative = float(absolute_values[values < 0].sum() / total_mass)
    return {
        'q_positive': q_positive,
        'q_negative': q_negative,
        'dominance': max(q_positive, q_negative),
        'dominant_sign': 1 if q_positive >= q_negative else -1,
    }


def simple_tree_cubic_curvature_features(
    x: np.ndarray, y: np.ndarray
) -> dict:
    result = {
        'fit_converged': False, 'fit_error': '',
        'a3': np.nan, 'a2': np.nan, 'a1': np.nan, 'a0': np.nan,
        'r2_diagnostic': np.nan,
        'f1_sign_changes': np.nan, 'f2_sign_changes': np.nan,
        'f2_q_positive': np.nan, 'f2_q_negative': np.nan,
        'f2_dominance': np.nan, 'f2_dominant_sign': 0,
        'inflection_x': np.nan, 'inflection_interior': False,
    }
    try:
        order = np.argsort(x)
        x_sorted = np.asarray(x, dtype=float)[order]
        y_sorted = np.asarray(y, dtype=float)[order]
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            coefficients = np.polyfit(x_sorted, y_sorted, 3)
        polynomial = np.poly1d(coefficients)
        prediction = polynomial(x_sorted)
        x_range = float(x_sorted[-1] - x_sorted[0])
        if not np.isfinite(x_range) or x_range <= np.finfo(float).eps:
            raise ValueError('Cubic curvature fit requires non-constant x.')
        x_lower = (
            x_sorted[0] + SIMPLE_TREE_CUBIC_INTERIOR_MARGIN * x_range
        )
        x_upper = (
            x_sorted[-1] - SIMPLE_TREE_CUBIC_INTERIOR_MARGIN * x_range
        )
        grid = np.linspace(
            x_lower, x_upper, SIMPLE_TREE_CUBIC_GRID_SIZE
        )
        first_derivative = np.polyder(polynomial, 1)(grid)
        second_derivative = np.polyder(polynomial, 2)(grid)
        second = simple_tree_signed_absolute_dominance(second_derivative)
        a3, a2, a1, a0 = map(float, coefficients)
        inflection_x = (
            float(-a2 / (3.0 * a3))
            if abs(a3) > np.finfo(float).eps else np.nan
        )
        result.update({
            'fit_converged': True,
            'a3': a3, 'a2': a2, 'a1': a1, 'a0': a0,
            'r2_diagnostic': float(r2_score(y_sorted, prediction)),
            'f1_sign_changes': simple_tree_count_sign_changes(
                first_derivative
            ),
            'f2_sign_changes': simple_tree_count_sign_changes(
                second_derivative
            ),
            'f2_q_positive': second['q_positive'],
            'f2_q_negative': second['q_negative'],
            'f2_dominance': second['dominance'],
            'f2_dominant_sign': second['dominant_sign'],
            'inflection_x': inflection_x,
            'inflection_interior': bool(
                np.isfinite(inflection_x)
                and x_lower <= inflection_x <= x_upper
            ),
        })
    except Exception as error:
        result['fit_error'] = f'{type(error).__name__}: {error}'
    return result


shape_fit_indices = np.flatnonzero(
    simple_tree_results['remaining_shape_pool'].to_numpy()
)
print(f'Cubic curvature eligible: {len(shape_fit_indices):,}')

# Fit one Cubic model to every case in the merged non-linear shape pool.
shape_cubic_rows = [
    simple_tree_cubic_curvature_features(
        simple_tree_x[index], simple_tree_y[index]
    )
    for index in shape_fit_indices
]
simple_tree_results = simple_tree_attach_features(
    simple_tree_results, shape_fit_indices, shape_cubic_rows, 'shape_cubic_',
    'Not eligible: not in the non-linear Cubic shape pool',
)

remaining = simple_tree_results['remaining_shape_pool']
cubic_converged = simple_tree_results['shape_cubic_fit_converged']
cubic_eligible = remaining & cubic_converged

# Apply the strict first-derivative rule and weighted second-derivative rule.
cubic_no_f1_change = (
    simple_tree_results['shape_cubic_f1_sign_changes'].eq(0)
)
cubic_dominant_curvature = (
    simple_tree_results['shape_cubic_f2_dominance'].ge(
        SIMPLE_TREE_CURVATURE_DOMINANCE_THRESHOLD
    )
)
cubic_supported_inflection = (
    simple_tree_results['shape_cubic_f2_dominance'].lt(
        SIMPLE_TREE_CURVATURE_DOMINANCE_THRESHOLD
    )
    & simple_tree_results['shape_cubic_f2_sign_changes'].eq(1)
)

simple_tree_results['final_shape'] = 'Other/Uncertain'
simple_tree_results.loc[
    simple_tree_results['confirmed_linear'], 'final_shape'
] = 'Linear'
simple_tree_results.loc[
    cubic_eligible & cubic_no_f1_change & cubic_dominant_curvature
    & simple_tree_results['shape_cubic_f2_dominant_sign'].lt(0),
    'final_shape',
] = 'Concave'
simple_tree_results.loc[
    cubic_eligible & cubic_no_f1_change & cubic_dominant_curvature
    & simple_tree_results['shape_cubic_f2_dominant_sign'].gt(0),
    'final_shape',
] = 'Convex'
simple_tree_results.loc[
    cubic_eligible & cubic_no_f1_change & cubic_supported_inflection,
    'final_shape',
] = 'S-shaped'

simple_tree_results['shape_decision_reason'] = 'Not in non-linear pool'
simple_tree_results.loc[
    simple_tree_results['confirmed_linear'], 'shape_decision_reason'
] = 'Power-law |b-1| within Linear tolerance'
simple_tree_results.loc[
    remaining & ~cubic_converged,
    'shape_decision_reason',
] = 'Cubic fit unavailable'
simple_tree_results.loc[
    cubic_eligible & ~cubic_no_f1_change, 'shape_decision_reason'
] = 'Cubic fit; first derivative changes sign'
simple_tree_results.loc[
    cubic_eligible & simple_tree_results['final_shape'].eq('Concave'),
    'shape_decision_reason',
] = 'Cubic fit; dominant negative second derivative'
simple_tree_results.loc[
    cubic_eligible & simple_tree_results['final_shape'].eq('Convex'),
    'shape_decision_reason',
] = 'Cubic fit; dominant positive second derivative'
simple_tree_results.loc[
    cubic_eligible & simple_tree_results['final_shape'].eq('S-shaped'),
    'shape_decision_reason',
] = 'Cubic fit; supported curvature sign change'

simple_tree_results['shape_fit_converged'] = cubic_converged

# Routing invariants.
complex_mask = ~simple_tree_results['simple_relationship']
if (complex_mask & simple_tree_results['linear_fit_converged']).any():
    raise AssertionError('Complex cases must not enter the Linear fit.')
if (complex_mask & simple_tree_results['shape_fit_converged']).any():
    raise AssertionError('Complex cases must not enter Cubic curvature fitting.')
branch_total = (
    simple_tree_results['shape_branch_1'].astype(int)
    + simple_tree_results['shape_branch_2'].astype(int)
    + simple_tree_results['shape_branch_3'].astype(int)
)
if not branch_total.eq(
    simple_tree_results['remaining_shape_pool'].astype(int)
).all():
    raise AssertionError(
        'Non-linear pool must equal the three disjoint failure branches.'
    )
if not simple_tree_results['final_shape'].eq('Linear').eq(
    simple_tree_results['confirmed_linear']
).all():
    raise AssertionError('Only confirmed Linear cases may terminate as Linear.')

print('Cubic curvature fits:')
print(pd.Series({
    'Eligible': int(remaining.sum()),
    'Converged': int(cubic_eligible.sum()),
    'Unavailable': int((remaining & ~cubic_converged).sum()),
}).to_string())
print('\nFinal shape counts:')
print(simple_tree_results['final_shape'].value_counts().to_string())

Cubic curvature eligible: 15,792


Cubic curvature fits:
Eligible       15792
Converged      15792
Unavailable        0

Final shape counts:
final_shape
Other/Uncertain    11663
Concave             4467
S-shaped            3075
Linear              2877
Convex              2561


In [19]:
# ============================================================
# STEP 8 CONFIG — EXPECTED LABELS AND PRINTED TABLES
# ============================================================
SIMPLE_TREE_FAMILY_ORDER = [
    'F01', 'F03', 'F05', 'F07', 'F13', 'F15', 'F18',
    'F19', 'F21', 'F22', 'F23', 'F24', 'F25', 'F26',
]
SIMPLE_TREE_FAMILY_NAMES = {
    'F01': 'Linear positive',
    'F03': 'Power convex positive',
    'F05': 'Power concave positive',
    'F07': 'Saturation positive',
    'F13': 'S-curve positive',
    'F15': 'Threshold positive',
    'F18': 'U-shape',
    'F19': 'Spike',
    'F21': 'Cubic',
    'F22': 'Oscillation / complex non-monotonic',
    'F23': 'Two Lines',
    'F24': 'Line + Parabola',
    'F25': 'Multi-regime',
    'F26': 'Windowed J',
}
SIMPLE_TREE_EXPECTED_CLASS_BY_FAMILY = {
    'F01': 'Linear',
    'F03': 'Convex',
    'F05': 'Concave',
    'F07': 'Concave',
    'F13': 'S-shaped',
    'F15': 'Other/Uncertain',
    'F18': 'Other/Uncertain',
    'F19': 'Other/Uncertain',
    'F21': 'Other/Uncertain',
    'F22': 'Other/Uncertain',
    'F23': 'Other/Uncertain',
    'F24': 'Other/Uncertain',
    'F25': 'Other/Uncertain',
    'F26': 'Other/Uncertain',
}
SIMPLE_TREE_CLASS_ORDER = [
    'Linear', 'Concave', 'Convex', 'S-shaped', 'Other/Uncertain',
]
SIMPLE_TREE_REPORT_ORDER = ['strong', 'mild', 'standard', 'combined']

simple_tree_results['expected_shape'] = simple_tree_results['family_id'].map(
    SIMPLE_TREE_EXPECTED_CLASS_BY_FAMILY
)
simple_tree_results['shape_correct'] = simple_tree_results['final_shape'].eq(
    simple_tree_results['expected_shape']
)
combined_simple_tree = simple_tree_results.copy()
combined_simple_tree['report_variant'] = 'combined'
simple_tree_results['report_variant'] = simple_tree_results['variant_level']
simple_tree_report_source = pd.concat(
    [simple_tree_results, combined_simple_tree], ignore_index=True
)

simple_tree_overall_rows = []
simple_tree_routing_rows = []
simple_tree_formatted_tables = {}
for variant in SIMPLE_TREE_REPORT_ORDER:
    variant_results = simple_tree_report_source.loc[
        simple_tree_report_source['report_variant'].eq(variant)
    ].copy()
    n_cases = len(variant_results)
    n_correct = int(variant_results['shape_correct'].sum())
    simple_tree_overall_rows.append({
        'variant': variant,
        'n_cases': n_cases,
        'n_correct': n_correct,
        'accuracy': n_correct / n_cases if n_cases else np.nan,
    })
    simple_tree_routing_rows.append({
        'variant': variant,
        'mic_ge_0p8': n_cases,
        'simple': int(variant_results['simple_relationship'].sum()),
        'complex': int((~variant_results['simple_relationship']).sum()),
        'strong_monotonic': int(variant_results['strong_monotonic'].sum()),
        'strong_linearity_candidate': int(
            variant_results['strong_linearity_candidate'].sum()
        ),
        'confirmed_linear': int(variant_results['confirmed_linear'].sum()),
        'branch1_not_strong_monotonic': int(
            variant_results['shape_branch_1'].sum()
        ),
        'branch2_not_strong_linearity': int(
            variant_results['shape_branch_2'].sum()
        ),
        'branch3_not_linear_by_powerlaw': int(
            variant_results['shape_branch_3'].sum()
        ),
        'remaining_shape_pool': int(
            variant_results['remaining_shape_pool'].sum()
        ),
        'cubic_fit_converged': int(
            variant_results['remaining_shape_pool'].mul(
                variant_results['shape_cubic_fit_converged']
            ).sum()
        ),
    })

    counts = pd.crosstab(
        variant_results['family_id'], variant_results['final_shape']
    ).reindex(
        index=SIMPLE_TREE_FAMILY_ORDER,
        columns=SIMPLE_TREE_CLASS_ORDER,
        fill_value=0,
    )
    table = pd.DataFrame({
        'family_id': SIMPLE_TREE_FAMILY_ORDER,
        'family_name': [
            SIMPLE_TREE_FAMILY_NAMES[family_id]
            for family_id in SIMPLE_TREE_FAMILY_ORDER
        ],
    })
    totals = counts.sum(axis=1)
    for label in SIMPLE_TREE_CLASS_ORDER:
        table[label] = [
            f'{int(count)} ({count / total:.1%})' if total else '0 (N/A)'
            for count, total in zip(counts[label], totals)
        ]
    table['Total'] = totals.to_numpy(dtype=int)
    correct_counts = []
    for family_id in SIMPLE_TREE_FAMILY_ORDER:
        expected = SIMPLE_TREE_EXPECTED_CLASS_BY_FAMILY[family_id]
        correct_counts.append(int(counts.loc[family_id, expected]))
    table['Accuracy'] = [
        f'{correct}/{int(total)} ({correct / total:.1%})'
        if total else 'N/A'
        for correct, total in zip(correct_counts, totals)
    ]
    simple_tree_formatted_tables[variant] = table
    table.to_csv(
        SIMPLE_TREE_OUTPUT_DIR / f'{variant}_classification_table.csv',
        index=False,
    )

simple_tree_overall_accuracy = pd.DataFrame(simple_tree_overall_rows)
simple_tree_routing_summary = pd.DataFrame(simple_tree_routing_rows)
simple_tree_results.to_parquet(
    SIMPLE_TREE_OUTPUT_DIR / 'simple_hierarchy_fits_full.parquet', index=False
)
simple_tree_overall_accuracy.to_csv(
    SIMPLE_TREE_OUTPUT_DIR / 'overall_accuracy.csv', index=False
)
simple_tree_routing_summary.to_csv(
    SIMPLE_TREE_OUTPUT_DIR / 'routing_summary.csv', index=False
)

print('Routing summary:')
print(simple_tree_routing_summary.to_string(index=False))
print('\nOverall accuracy:')
print(
    simple_tree_overall_accuracy.assign(
        accuracy=simple_tree_overall_accuracy['accuracy'].map('{:.2%}'.format)
    ).to_string(index=False)
)
# Split the 14 family rows into two side-by-side halves.  Each half keeps
# the complete schema, so the output is half as tall without omitting data.
def simple_tree_side_by_side(table):
    split_at = int(np.ceil(len(table) / 2))
    left_lines = table.iloc[:split_at].to_string(index=False).splitlines()
    right_lines = table.iloc[split_at:].to_string(index=False).splitlines()
    left_width = max(len(line) for line in left_lines)
    n_lines = max(len(left_lines), len(right_lines))
    left_lines += [''] * (n_lines - len(left_lines))
    right_lines += [''] * (n_lines - len(right_lines))
    return '\n'.join(
        f'{left.ljust(left_width)}    |    {right}'
        for left, right in zip(left_lines, right_lines)
    )

# ============================================================
# STEP 8 PRINT CONFIG — DECISION TABLES AT EVERY TREE STAGE
# ============================================================
SIMPLE_TREE_PRINT_DECISION_STEP_TABLES = True
SIMPLE_TREE_SAVE_DECISION_STEP_TABLES = True


def simple_tree_variant_slice(frame, variant):
    if variant == 'combined':
        return frame.copy()
    return frame.loc[frame['variant_level'].eq(variant)].copy()


def simple_tree_empty_family_table():
    return pd.DataFrame({
        'family_id': SIMPLE_TREE_FAMILY_ORDER,
        'family_name': [
            SIMPLE_TREE_FAMILY_NAMES[family_id]
            for family_id in SIMPLE_TREE_FAMILY_ORDER
        ],
    })


def simple_tree_count_and_percent(count, denominator):
    if denominator:
        return f'{int(count)} ({count / denominator:.1%})'
    return f'{int(count)} (N/A)'


def simple_tree_make_binary_step_table(
    variant_results,
    eligible_mask,
    pass_mask,
    eligible_label,
    pass_label,
    fail_label,
    pass_rate_label='Pass rate',
):
    table = simple_tree_empty_family_table()
    input_values = []
    eligible_values = []
    pass_values = []
    fail_values = []
    pass_rate_values = []
    included_family_ids = []
    for family_id in SIMPLE_TREE_FAMILY_ORDER:
        family_mask = variant_results['family_id'].eq(family_id)
        n_input = int(family_mask.sum())
        n_eligible = int((family_mask & eligible_mask).sum())
        if n_eligible == 0:
            continue
        included_family_ids.append(family_id)
        n_pass = int((family_mask & eligible_mask & pass_mask).sum())
        n_fail = n_eligible - n_pass
        input_values.append(int(n_input))
        eligible_values.append(
            simple_tree_count_and_percent(n_eligible, n_eligible)
        )
        pass_values.append(
            simple_tree_count_and_percent(n_pass, n_eligible)
        )
        fail_values.append(
            simple_tree_count_and_percent(n_fail, n_eligible)
        )
        pass_rate_values.append(
            f'{n_pass}/{n_eligible} ({n_pass / n_eligible:.1%})'
            if n_eligible else 'N/A'
        )
    table = table.loc[
        table['family_id'].isin(included_family_ids)
    ].reset_index(drop=True)
    table[eligible_label] = eligible_values
    table[pass_label] = pass_values
    table[fail_label] = fail_values
    table[pass_rate_label] = pass_rate_values
    return table


def simple_tree_make_shape_pool_table(variant_results):
    table = simple_tree_empty_family_table()
    shape_pool = variant_results['remaining_shape_pool']
    input_values = []
    pool_values = []
    fit_values = []
    unavailable_values = []
    shape_values = {label: [] for label in SIMPLE_TREE_CLASS_ORDER[1:]}
    included_family_ids = []
    for family_id in SIMPLE_TREE_FAMILY_ORDER:
        family_mask = variant_results['family_id'].eq(family_id)
        family_pool = family_mask & shape_pool
        n_input = int(family_mask.sum())
        n_pool = int(family_pool.sum())
        if n_pool == 0:
            continue
        included_family_ids.append(family_id)
        n_fit = int(
            (family_pool & variant_results['shape_fit_converged']).sum()
        )
        input_values.append(n_input)
        pool_values.append(
            simple_tree_count_and_percent(n_pool, n_pool)
        )
        fit_values.append(
            simple_tree_count_and_percent(n_fit, n_pool)
        )
        n_unavailable = n_pool - n_fit
        unavailable_values.append(
            simple_tree_count_and_percent(n_unavailable, n_pool)
        )
        for label in SIMPLE_TREE_CLASS_ORDER[1:]:
            n_label = int(
                (family_pool & variant_results['final_shape'].eq(label)).sum()
            )
            shape_values[label].append(
                simple_tree_count_and_percent(n_label, n_pool)
            )
    table = table.loc[
        table['family_id'].isin(included_family_ids)
    ].reset_index(drop=True)
    table['Shape pool (Branch 1+2+3)'] = pool_values
    table['Cubic fit converged'] = fit_values
    table['Cubic fit unavailable'] = unavailable_values
    for label in SIMPLE_TREE_CLASS_ORDER[1:]:
        table[label] = shape_values[label]
    return table


# Step 1 uses all three mutually exclusive S4 MIC partitions so its pass
# percentage is relative to every S4 case, not just the already-filtered input.
simple_tree_mic_partition_frames = []
for mic_filename, mic_passes in [
    ('mic_0_to_0p6_full.parquet', False),
    ('mic_0p6_to_0p8_full.parquet', False),
    ('mic_0p8_to_1_full.parquet', True),
]:
    mic_path = S4_DIR / mic_filename
    if not mic_path.exists():
        raise FileNotFoundError(mic_path)
    mic_frame = pd.read_parquet(
        mic_path, columns=['family_id', 'variant_level']
    )
    mic_frame['mic_ge_0p8'] = mic_passes
    simple_tree_mic_partition_frames.append(mic_frame)
simple_tree_all_mic_cases = pd.concat(
    simple_tree_mic_partition_frames, ignore_index=True
)

# Remove superseded decision tables whose meanings were merged/renumbered.
for obsolete_variant in SIMPLE_TREE_REPORT_ORDER:
    for obsolete_step in [
        'step5_power_fit',
        'step6_linear_confirmation',
        'step7_remaining_shapes',
        'step4_pearson_gate',
        'step5_power_law_linear_other',
        'step6_remaining_shapes',
    ]:
        obsolete_path = SIMPLE_TREE_OUTPUT_DIR / (
            f'{obsolete_variant}_{obsolete_step}.csv'
        )
        obsolete_path.unlink(missing_ok=True)

simple_tree_step_tables = {}
for variant in SIMPLE_TREE_REPORT_ORDER:
    variant_results = simple_tree_variant_slice(simple_tree_results, variant)
    all_variant_mic = simple_tree_variant_slice(
        simple_tree_all_mic_cases, variant
    )

    # Step 1: MIC >= 0.8 selection relative to every S4 case.
    step1 = simple_tree_empty_family_table()
    all_counts = all_variant_mic.groupby('family_id').size().reindex(
        SIMPLE_TREE_FAMILY_ORDER, fill_value=0
    )
    mic_pass_counts = all_variant_mic.loc[
        all_variant_mic['mic_ge_0p8']
    ].groupby('family_id').size().reindex(
        SIMPLE_TREE_FAMILY_ORDER, fill_value=0
    )
    mic_fail_counts = all_counts - mic_pass_counts
    step1['All S4 cases'] = all_counts.to_numpy(dtype=int)
    step1['MIC>=0.8'] = [
        simple_tree_count_and_percent(count, total)
        for count, total in zip(mic_pass_counts, all_counts)
    ]
    step1['MIC<0.8'] = [
        simple_tree_count_and_percent(count, total)
        for count, total in zip(mic_fail_counts, all_counts)
    ]
    step1['Pass rate'] = [
        f'{int(count)}/{int(total)} ({count / total:.1%})'
        if total else 'N/A'
        for count, total in zip(mic_pass_counts, all_counts)
    ]

    all_eligible = pd.Series(True, index=variant_results.index)
    step2 = simple_tree_make_binary_step_table(
        variant_results,
        all_eligible,
        variant_results['simple_relationship'],
        'Eligible MIC>=0.8',
        'Simple',
        'Complex',
    )
    step3 = simple_tree_make_binary_step_table(
        variant_results,
        variant_results['simple_relationship'],
        variant_results['strong_monotonic'],
        'Eligible Simple',
        'Strong monotonic',
        'Not strong monotonic within Simple',
    )
    step4 = simple_tree_make_binary_step_table(
        variant_results,
        variant_results['strong_monotonic'],
        variant_results['confirmed_linear'],
        'Eligible strong monotonic',
        'Linear',
        'Other / sent to shape pool',
        'Linear rate',
    )
    step5 = simple_tree_make_shape_pool_table(variant_results)

    simple_tree_step_tables[variant] = {
        'step1_mic_selection': step1,
        'step2_simple_complex': step2,
        'step3_strong_monotonicity': step3,
        'step4_power_law_linear_other': step4,
        'step5_remaining_shapes': step5,
    }
    if SIMPLE_TREE_SAVE_DECISION_STEP_TABLES:
        for step_key, step_table in simple_tree_step_tables[variant].items():
            step_table.to_csv(
                SIMPLE_TREE_OUTPUT_DIR / f'{variant}_{step_key}.csv',
                index=False,
            )

SIMPLE_TREE_STEP_TITLES = {
    'step1_mic_selection': 'Step 1 — MIC >= 0.8 selection',
    'step2_simple_complex': 'Step 2 — Simple / Complex',
    'step3_strong_monotonicity': 'Step 3 — Strong monotonicity within Simple',
    'step4_power_law_linear_other': 'Step 4 — Free power-law: Linear / Other',
    'step5_remaining_shapes': 'Step 5 — Cubic-only curvature classification',
}

for variant in SIMPLE_TREE_REPORT_ORDER:
    if SIMPLE_TREE_PRINT_DECISION_STEP_TABLES:
        print(f'\n{variant.title()} — decision flow by family:')
        for step_key, step_table in simple_tree_step_tables[variant].items():
            print(f'\n{SIMPLE_TREE_STEP_TITLES[step_key]}')
            print(simple_tree_side_by_side(step_table))
    print(f'\n{variant.title()} — simplified hierarchy:')
    variant_table = simple_tree_formatted_tables[variant]
    print(simple_tree_side_by_side(variant_table))
# ============================================================
# GLOBAL FINAL TABLE — STRONG VARIANT, ALL MIC >= 0.8 FAMILIES
# ============================================================
SIMPLE_TREE_GLOBAL_TABLE_VARIANT = 'strong'
global_results = simple_tree_results.loc[
    simple_tree_results['variant_level'].eq(
        SIMPLE_TREE_GLOBAL_TABLE_VARIANT
    )
].copy()
global_counts = pd.crosstab(
    global_results['family_id'], global_results['final_shape']
).reindex(
    index=SIMPLE_TREE_FAMILY_ORDER,
    columns=SIMPLE_TREE_CLASS_ORDER,
    fill_value=0,
)
global_totals = global_counts.sum(axis=1)
simple_tree_global_table = pd.DataFrame({
    'family_id': SIMPLE_TREE_FAMILY_ORDER,
    'family_name': [
        SIMPLE_TREE_FAMILY_NAMES[family_id]
        for family_id in SIMPLE_TREE_FAMILY_ORDER
    ],
    'MIC≥0.8 input': global_totals.to_numpy(dtype=int),
})
for label in SIMPLE_TREE_CLASS_ORDER:
    simple_tree_global_table[label] = [
        f'{int(count)} ({count / total:.1%})' if count and total else '0'
        for count, total in zip(global_counts[label], global_totals)
    ]
simple_tree_global_table.to_csv(
    SIMPLE_TREE_OUTPUT_DIR / 'global_strong_classification_table.csv',
    index=False,
)

print('\nGlobal final classification table — Strong variant:')
print(simple_tree_global_table.to_string(index=False))
print(f'\nSaved: {SIMPLE_TREE_OUTPUT_DIR.relative_to(REPO_ROOT)}')

Routing summary:
 variant  mic_ge_0p8  simple  complex  strong_monotonic  strong_linearity_candidate  confirmed_linear  branch1_not_strong_monotonic  branch2_not_strong_linearity  branch3_not_linear_by_powerlaw  remaining_shape_pool  cubic_fit_converged
  strong        7457    5091     2366              3279                        3279               962                          1812                             0                            2317                  4129                 4129
    mild        8706    7498     1208              5285                        5285               990                          2213                             0                            4295                  6508                 6508
standard        8480    6080     2400              4344                        4344               925                          1736                             0                            3419                  5155                 5155
combined       24643   18669     59


Strong — decision flow by family:

Step 1 — MIC >= 0.8 selection
family_id            family_name  All S4 cases     MIC>=0.8      MIC<0.8         Pass rate    |    family_id                         family_name  All S4 cases    MIC>=0.8       MIC<0.8        Pass rate
      F01        Linear positive          2270  994 (43.8%) 1276 (56.2%)  994/2270 (43.8%)    |          F19                               Spike          2270    0 (0.0%) 2270 (100.0%)    0/2270 (0.0%)
      F03  Power convex positive          2270  806 (35.5%) 1464 (64.5%)  806/2270 (35.5%)    |          F21                               Cubic          2270 549 (24.2%)  1721 (75.8%) 549/2270 (24.2%)
      F05 Power concave positive          2270  856 (37.7%) 1414 (62.3%)  856/2270 (37.7%)    |          F22 Oscillation / complex non-monotonic          2270 987 (43.5%)  1283 (56.5%) 987/2270 (43.5%)
      F07    Saturation positive          2270    55 (2.4%) 2215 (97.6%)    55/2270 (2.4%)    |          F23                  

# Additional hierarchical S5 — 0.6 ≤ MIC < 0.8 full cohort

This section applies the exact same sequential exclusions and strict
Cubic-only curvature rule to the intermediate MIC cohort. It does not replace
or overwrite the primary MIC ≥ 0.8 analysis. Step 1 through Step 6 and the
final classification table are printed separately for Strong, Mild,
Standard, and Combined.

In [20]:
# ============================================================
# STEP 1 CONFIG — INTERMEDIATE MIC INPUT / OUTPUT
# ============================================================
MID_TREE_INPUT_FILENAME = 'mic_0p6_to_0p8_full.parquet'
MID_TREE_OUTPUT_SUBDIR = 'simple_hierarchy_mic_0p6_to_0p8'
MID_TREE_INPUT_LABEL = '0.6<=MIC<0.8 input'
MID_TREE_OUTPUT_DIR = OUTPUT_DIR / MID_TREE_OUTPUT_SUBDIR
MID_TREE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for obsolete_variant in SIMPLE_TREE_REPORT_ORDER:
    for obsolete_step in [
        'step4_pearson_gate',
        'step5_power_law_linear_other',
        'step6_remaining_shapes',
    ]:
        (MID_TREE_OUTPUT_DIR / f'{obsolete_variant}_{obsolete_step}.csv').unlink(
            missing_ok=True
        )

mid_tree_cases = pd.read_parquet(S4_DIR / MID_TREE_INPUT_FILENAME)
mid_tree_cases = mid_tree_cases.sort_values(
    'candidate_index'
).reset_index(drop=True)
mid_tree_indices = mid_tree_cases['candidate_index'].to_numpy(dtype=int)
mid_tree_x = points['x'][mid_tree_indices].astype(float)
mid_tree_y = points['y'][mid_tree_indices].astype(float)
print(f'0.6 <= MIC < 0.8 input: {len(mid_tree_cases):,} cases')

# ============================================================
# STEP 2 CONFIG — SIMPLE / COMPLEX
# ============================================================
MID_TREE_MIN_ABS_CORRELATION_THRESHOLD = 0.70
mid_tree_cases['simple_relationship'] = (
    mid_tree_cases['pearson_r'].abs().ge(
        MID_TREE_MIN_ABS_CORRELATION_THRESHOLD
    )
    & mid_tree_cases['spearman_rho'].abs().ge(
        MID_TREE_MIN_ABS_CORRELATION_THRESHOLD
    )
)
mid_tree_cases['simple_complex_label'] = np.where(
    mid_tree_cases['simple_relationship'], 'Simple', 'Complex'
)

# ============================================================
# STEP 3 CONFIG — STRONG MONOTONICITY WITHIN SIMPLE
# ============================================================
MID_TREE_STRONG_MONOTONIC_SPEARMAN_THRESHOLD = 0.90
mid_tree_cases['strong_monotonic'] = (
    mid_tree_cases['simple_relationship']
    & mid_tree_cases['spearman_rho'].abs().ge(
        MID_TREE_STRONG_MONOTONIC_SPEARMAN_THRESHOLD
    )
)
mid_tree_cases['simple_not_strong_monotonic'] = (
    mid_tree_cases['simple_relationship']
    & ~mid_tree_cases['strong_monotonic']
)

# ============================================================
# STEP 4 CONFIG — STRONG LINEARITY WITHIN STRONG MONOTONIC
# ============================================================
MID_TREE_USE_STRONG_LINEARITY_GATE = False
MID_TREE_STRONG_LINEARITY_PEARSON_THRESHOLD = 0.90
if MID_TREE_USE_STRONG_LINEARITY_GATE:
    mid_tree_cases['strong_linearity_candidate'] = (
        mid_tree_cases['strong_monotonic']
        & mid_tree_cases['pearson_r'].abs().gt(
            MID_TREE_STRONG_LINEARITY_PEARSON_THRESHOLD
        )
    )
else:
    # Pearson gate OFF: every Strong-monotonic case proceeds directly
    # to the free power-law Linear confirmation.
    mid_tree_cases['strong_linearity_candidate'] = mid_tree_cases[
        'strong_monotonic'
    ].copy()
mid_tree_cases['strong_monotonic_not_strong_linearity'] = (
    mid_tree_cases['strong_monotonic']
    & ~mid_tree_cases['strong_linearity_candidate']
)

# ============================================================
# STEP 5 CONFIG — FREE POWER-LAW LINEAR CONFIRMATION
# ============================================================
MID_TREE_POWER_EXPONENT_LINEAR_TOLERANCE = 0.05
mid_tree_results = mid_tree_cases.copy()
mid_linear_indices = np.flatnonzero(
    mid_tree_results['strong_linearity_candidate'].to_numpy()
)
mid_linear_rows = [
    simple_tree_power_features(mid_tree_x[index], mid_tree_y[index])
    for index in mid_linear_indices
]
mid_tree_results = simple_tree_attach_features(
    mid_tree_results, mid_linear_indices, mid_linear_rows, 'linear_',
    'Not eligible: did not pass Strong monotonicity',
)
mid_tree_results['linear_abs_b_minus_one'] = (
    mid_tree_results['linear_b'] - 1.0
).abs()
mid_tree_results['confirmed_linear'] = (
    mid_tree_results['strong_linearity_candidate']
    & mid_tree_results['linear_fit_converged']
    & mid_tree_results['linear_abs_b_minus_one'].le(
        MID_TREE_POWER_EXPONENT_LINEAR_TOLERANCE
    )
)
mid_tree_results['strong_linearity_not_linear'] = (
    mid_tree_results['strong_linearity_candidate']
    & ~mid_tree_results['confirmed_linear']
)
mid_tree_results['shape_branch_1'] = (
    mid_tree_results['simple_not_strong_monotonic']
)
mid_tree_results['shape_branch_2'] = (
    mid_tree_results['strong_monotonic_not_strong_linearity']
)
mid_tree_results['shape_branch_3'] = (
    mid_tree_results['strong_linearity_not_linear']
)
mid_tree_results['remaining_shape_pool'] = (
    mid_tree_results['shape_branch_1']
    | mid_tree_results['shape_branch_2']
    | mid_tree_results['shape_branch_3']
)

# ============================================================
# STEP 6 CONFIG — STRICT CUBIC-ONLY CURVATURE
# ============================================================
MID_TREE_CURVATURE_DOMINANCE_THRESHOLD = 0.75
mid_shape_indices = np.flatnonzero(
    mid_tree_results['remaining_shape_pool'].to_numpy()
)
mid_cubic_rows = [
    simple_tree_cubic_curvature_features(
        mid_tree_x[index], mid_tree_y[index]
    )
    for index in mid_shape_indices
]
mid_tree_results = simple_tree_attach_features(
    mid_tree_results, mid_shape_indices, mid_cubic_rows, 'shape_cubic_',
    'Not eligible: not in the non-linear Cubic shape pool',
)
mid_remaining = mid_tree_results['remaining_shape_pool']
mid_cubic_converged = mid_tree_results['shape_cubic_fit_converged']
mid_cubic_eligible = mid_remaining & mid_cubic_converged
mid_cubic_no_f1_change = (
    mid_tree_results['shape_cubic_f1_sign_changes'].eq(0)
)
mid_cubic_dominant = (
    mid_tree_results['shape_cubic_f2_dominance'].ge(
        MID_TREE_CURVATURE_DOMINANCE_THRESHOLD
    )
)
mid_cubic_inflection = (
    mid_tree_results['shape_cubic_f2_dominance'].lt(
        MID_TREE_CURVATURE_DOMINANCE_THRESHOLD
    )
    & mid_tree_results['shape_cubic_f2_sign_changes'].eq(1)
)
mid_tree_results['final_shape'] = 'Other/Uncertain'
mid_tree_results.loc[
    mid_tree_results['confirmed_linear'], 'final_shape'
] = 'Linear'
mid_tree_results.loc[
    mid_cubic_eligible & mid_cubic_no_f1_change & mid_cubic_dominant
    & mid_tree_results['shape_cubic_f2_dominant_sign'].lt(0),
    'final_shape',
] = 'Concave'
mid_tree_results.loc[
    mid_cubic_eligible & mid_cubic_no_f1_change & mid_cubic_dominant
    & mid_tree_results['shape_cubic_f2_dominant_sign'].gt(0),
    'final_shape',
] = 'Convex'
mid_tree_results.loc[
    mid_cubic_eligible & mid_cubic_no_f1_change & mid_cubic_inflection,
    'final_shape',
] = 'S-shaped'
mid_tree_results['shape_fit_converged'] = mid_cubic_converged
mid_tree_results['expected_shape'] = mid_tree_results['family_id'].map(
    SIMPLE_TREE_EXPECTED_CLASS_BY_FAMILY
)
mid_tree_results['shape_correct'] = mid_tree_results['final_shape'].eq(
    mid_tree_results['expected_shape']
)

# Routing invariants.
mid_complex = ~mid_tree_results['simple_relationship']
if (mid_complex & mid_tree_results['linear_fit_converged']).any():
    raise AssertionError('Complex cases entered the Linear fit.')
if (mid_complex & mid_tree_results['shape_fit_converged']).any():
    raise AssertionError('Complex cases entered Cubic fitting.')
if not mid_tree_results['final_shape'].eq('Linear').eq(
    mid_tree_results['confirmed_linear']
).all():
    raise AssertionError('Only confirmed Linear cases may be Linear.')

# ============================================================
# STEP TABLES AND FINAL TABLES — STRONG / MILD / STANDARD / COMBINED
# ============================================================
mid_combined = mid_tree_results.copy()
mid_combined['report_variant'] = 'combined'
mid_tree_results['report_variant'] = mid_tree_results['variant_level']
mid_report_source = pd.concat(
    [mid_tree_results, mid_combined], ignore_index=True
)

def mid_relabel_input(table):
    return table.rename(columns={'MIC>=0.8 input': MID_TREE_INPUT_LABEL})


def mid_classification_table(variant_results):
    counts = pd.crosstab(
        variant_results['family_id'], variant_results['final_shape']
    ).reindex(
        index=SIMPLE_TREE_FAMILY_ORDER,
        columns=SIMPLE_TREE_CLASS_ORDER,
        fill_value=0,
    )
    totals = counts.sum(axis=1)
    table = simple_tree_empty_family_table()
    for label in SIMPLE_TREE_CLASS_ORDER:
        table[label] = [
            f'{int(count)} ({count / total:.1%})'
            if total else '0 (N/A)'
            for count, total in zip(counts[label], totals)
        ]
    table['Total'] = totals.to_numpy(dtype=int)
    correct = [
        int(counts.loc[family_id, SIMPLE_TREE_EXPECTED_CLASS_BY_FAMILY[family_id]])
        for family_id in SIMPLE_TREE_FAMILY_ORDER
    ]
    table['Accuracy'] = [
        f'{n_correct}/{int(total)} ({n_correct / total:.1%})'
        if total else 'N/A'
        for n_correct, total in zip(correct, totals)
    ]
    return table


# Step 1 denominator contains all three S4 MIC partitions.
mid_all_parts = []
for filename, is_mid in [
    ('mic_0_to_0p6_full.parquet', False),
    ('mic_0p6_to_0p8_full.parquet', True),
    ('mic_0p8_to_1_full.parquet', False),
]:
    part = pd.read_parquet(S4_DIR / filename)
    part['_mid_target'] = is_mid
    mid_all_parts.append(part)
mid_all_mic_cases = pd.concat(mid_all_parts, ignore_index=True)

MID_TREE_STEP_TITLES = {
    'step1_mic_selection': 'Step 1 — 0.6 <= MIC < 0.8 selection',
    'step2_simple_complex': 'Step 2 — Simple / Complex',
    'step3_strong_monotonicity': 'Step 3 — Strong monotonicity within Simple',
    'step4_power_law_linear_other': 'Step 4 — Free power-law: Linear / Other',
    'step5_remaining_shapes': 'Step 5 — Cubic-only curvature classification',
}
mid_step_tables = {}
mid_final_tables = {}
mid_overall_rows = []
mid_routing_rows = []

for variant in SIMPLE_TREE_REPORT_ORDER:
    variant_results = simple_tree_variant_slice(mid_tree_results, variant)
    all_variant = simple_tree_variant_slice(mid_all_mic_cases, variant)

    step1 = simple_tree_empty_family_table()
    all_counts = all_variant.groupby('family_id').size().reindex(
        SIMPLE_TREE_FAMILY_ORDER, fill_value=0
    )
    pass_counts = all_variant.loc[
        all_variant['_mid_target']
    ].groupby('family_id').size().reindex(
        SIMPLE_TREE_FAMILY_ORDER, fill_value=0
    )
    fail_counts = all_counts - pass_counts
    step1['All S4 cases'] = all_counts.to_numpy(dtype=int)
    step1['0.6<=MIC<0.8'] = [
        simple_tree_count_and_percent(n_pass, n_all)
        for n_pass, n_all in zip(pass_counts, all_counts)
    ]
    step1['Outside 0.6<=MIC<0.8'] = [
        simple_tree_count_and_percent(n_fail, n_all)
        for n_fail, n_all in zip(fail_counts, all_counts)
    ]
    step1['Pass rate'] = [
        f'{n_pass}/{n_all} ({n_pass / n_all:.1%})' if n_all else 'N/A'
        for n_pass, n_all in zip(pass_counts, all_counts)
    ]

    eligible_all = pd.Series(True, index=variant_results.index)
    step2 = mid_relabel_input(simple_tree_make_binary_step_table(
        variant_results, eligible_all,
        variant_results['simple_relationship'],
        'Eligible 0.6<=MIC<0.8', 'Simple', 'Complex', 'Pass rate',
    ))
    step3 = mid_relabel_input(simple_tree_make_binary_step_table(
        variant_results, variant_results['simple_relationship'],
        variant_results['strong_monotonic'],
        'Eligible Simple', 'Strong monotonic',
        'Not strong monotonic', 'Pass rate',
    ))
    step4 = mid_relabel_input(simple_tree_make_binary_step_table(
        variant_results,
        variant_results['strong_monotonic'],
        variant_results['confirmed_linear'],
        'Eligible strong monotonic', 'Linear',
        'Other / sent to shape pool', 'Linear rate',
    ))
    step5 = mid_relabel_input(
        simple_tree_make_shape_pool_table(variant_results)
    )
    mid_step_tables[variant] = {
        'step1_mic_selection': step1,
        'step2_simple_complex': step2,
        'step3_strong_monotonicity': step3,
        'step4_power_law_linear_other': step4,
        'step5_remaining_shapes': step5,
    }
    final_table = mid_classification_table(variant_results)
    mid_final_tables[variant] = final_table
    final_table.to_csv(
        MID_TREE_OUTPUT_DIR / f'{variant}_classification_table.csv',
        index=False,
    )
    for step_key, table in mid_step_tables[variant].items():
        table.to_csv(
            MID_TREE_OUTPUT_DIR / f'{variant}_{step_key}.csv',
            index=False,
        )

    n_cases = len(variant_results)
    n_correct = int(variant_results['shape_correct'].sum())
    mid_overall_rows.append({
        'variant': variant, 'n_cases': n_cases,
        'n_correct': n_correct,
        'accuracy': n_correct / n_cases if n_cases else np.nan,
    })
    mid_routing_rows.append({
        'variant': variant,
        'mic_0p6_to_0p8': n_cases,
        'simple': int(variant_results['simple_relationship'].sum()),
        'complex': int((~variant_results['simple_relationship']).sum()),
        'strong_monotonic': int(variant_results['strong_monotonic'].sum()),
        'strong_linearity_candidate': int(
            variant_results['strong_linearity_candidate'].sum()
        ),
        'confirmed_linear': int(variant_results['confirmed_linear'].sum()),
        'remaining_shape_pool': int(
            variant_results['remaining_shape_pool'].sum()
        ),
        'cubic_fit_converged': int((
            variant_results['remaining_shape_pool']
            & variant_results['shape_cubic_fit_converged']
        ).sum()),
    })

mid_overall_accuracy = pd.DataFrame(mid_overall_rows)
mid_routing_summary = pd.DataFrame(mid_routing_rows)
mid_tree_results.to_parquet(
    MID_TREE_OUTPUT_DIR / 'simple_hierarchy_fits_full.parquet', index=False
)
mid_overall_accuracy.to_csv(
    MID_TREE_OUTPUT_DIR / 'overall_accuracy.csv', index=False
)
mid_routing_summary.to_csv(
    MID_TREE_OUTPUT_DIR / 'routing_summary.csv', index=False
)

print('\n0.6 <= MIC < 0.8 routing summary:')
print(mid_routing_summary.to_string(index=False))
print('\n0.6 <= MIC < 0.8 overall accuracy:')
print(mid_overall_accuracy.assign(
    accuracy=mid_overall_accuracy['accuracy'].map('{:.2%}'.format)
).to_string(index=False))

for variant in SIMPLE_TREE_REPORT_ORDER:
    print(f'\n{variant.title()} — 0.6 <= MIC < 0.8 decision flow:')
    for step_key, table in mid_step_tables[variant].items():
        print(f'\n{MID_TREE_STEP_TITLES[step_key]}')
        print(simple_tree_side_by_side(table))
    print(f'\n{variant.title()} — 0.6 <= MIC < 0.8 final classification:')
    print(simple_tree_side_by_side(mid_final_tables[variant]))

print(f'\nSaved: {MID_TREE_OUTPUT_DIR.relative_to(REPO_ROOT)}')

0.6 <= MIC < 0.8 input: 9,567 cases



0.6 <= MIC < 0.8 routing summary:
 variant  mic_0p6_to_0p8  simple  complex  strong_monotonic  strong_linearity_candidate  confirmed_linear  remaining_shape_pool  cubic_fit_converged
  strong            2667     910     1757                65                          65                15                   895                  895
    mild            4060    1906     2154               107                         107                39                  1867                 1867
standard            2840    1216     1624                69                          69                12                  1204                 1204
combined            9567    4032     5535               241                         241                66                  3966                 3966

0.6 <= MIC < 0.8 overall accuracy:
 variant  n_cases  n_correct accuracy
  strong     2667       1725   64.68%
    mild     4060       3123   76.92%
standard     2840       2607   91.80%
combined     9567       7455   7

## Internal routing diagnostics (`0.6 <= MIC < 0.8`)

This block prepares internal routing checks but does not print the older Correct/Incorrect tables. The requested row-wise flow tables are shown in the next section.

In [21]:
# ============================================================
# STRONG-ONLY EXPECTED ROUTING — EDIT THESE SETS HERE
# ============================================================
STRONG_STEP_EXPECTED_SIMPLE_FAMILIES = {
    'F01', 'F03', 'F05', 'F07', 'F13',
}
STRONG_STEP_EXPECTED_STRONG_MONOTONIC_FAMILIES = {
    'F01', 'F03', 'F05', 'F07', 'F13',
}
STRONG_STEP_EXPECTED_STRONG_LINEAR_FAMILIES = {'F01'}
STRONG_STEP_EXPECTED_LINEAR_FAMILIES = {'F01'}

mid_strong_results = mid_tree_results.loc[
    mid_tree_results['variant_level'].eq('strong')
].copy()


def strong_step_binary_accuracy_table(
    results,
    eligible_mask,
    predicted_pass_mask,
    expected_pass_families,
    pass_label,
    fail_label,
):
    rows = []
    for family_id in SIMPLE_TREE_FAMILY_ORDER:
        family_mask = results['family_id'].eq(family_id)
        family_eligible = family_mask & eligible_mask
        n_input = int(family_mask.sum())
        n_eligible = int(family_eligible.sum())
        n_pass = int((family_eligible & predicted_pass_mask).sum())
        n_fail = n_eligible - n_pass
        expected_pass = family_id in expected_pass_families
        n_correct = n_pass if expected_pass else n_fail
        n_incorrect = n_eligible - n_correct
        rows.append({
            'family_id': family_id,
            'family_name': SIMPLE_TREE_FAMILY_NAMES[family_id],
            'Input': n_input,
            'Eligible': simple_tree_count_and_percent(
                n_eligible, n_input
            ),
            'Expected outcome': pass_label if expected_pass else fail_label,
            pass_label: simple_tree_count_and_percent(n_pass, n_eligible),
            fail_label: simple_tree_count_and_percent(n_fail, n_eligible),
            'Correct': simple_tree_count_and_percent(n_correct, n_eligible),
            'Incorrect': simple_tree_count_and_percent(
                n_incorrect, n_eligible
            ),
            'Accuracy': (
                f'{n_correct}/{n_eligible} '
                f'({n_correct / n_eligible:.1%})'
                if n_eligible else 'N/A'
            ),
        })
    return pd.DataFrame(rows)


# Step 1: selection only; it has no prediction target.
strong_accuracy_step1 = mid_step_tables['strong'][
    'step1_mic_selection'
].copy()

# Step 2: Simple / Complex among every Strong case in this MIC band.
strong_accuracy_step2 = strong_step_binary_accuracy_table(
    mid_strong_results,
    pd.Series(True, index=mid_strong_results.index),
    mid_strong_results['simple_relationship'],
    STRONG_STEP_EXPECTED_SIMPLE_FAMILIES,
    'Simple',
    'Complex',
)

# Step 3: Strong monotonicity only among cases that passed Step 2.
strong_accuracy_step3 = strong_step_binary_accuracy_table(
    mid_strong_results,
    mid_strong_results['simple_relationship'],
    mid_strong_results['strong_monotonic'],
    STRONG_STEP_EXPECTED_STRONG_MONOTONIC_FAMILIES,
    'Strong monotonic',
    'Not strong monotonic',
)

# Legacy diagnostic variable: Pearson gate is OFF, so this equals Step 3.
strong_accuracy_step4 = strong_step_binary_accuracy_table(
    mid_strong_results,
    mid_strong_results['strong_monotonic'],
    mid_strong_results['strong_linearity_candidate'],
    STRONG_STEP_EXPECTED_STRONG_LINEAR_FAMILIES,
    'Strong linearity',
    'Not strong linearity',
)

# Step 5: Power-law Linear confirmation among Step-4 pass cases.
strong_accuracy_step5 = strong_step_binary_accuracy_table(
    mid_strong_results,
    mid_strong_results['strong_linearity_candidate'],
    mid_strong_results['confirmed_linear'],
    STRONG_STEP_EXPECTED_LINEAR_FAMILIES,
    'Linear',
    'Not Linear',
)

# Step 6: shape accuracy among the merged Branch 1+2+3 shape pool.
strong_accuracy_step6_rows = []
for family_id in SIMPLE_TREE_FAMILY_ORDER:
    family_mask = mid_strong_results['family_id'].eq(family_id)
    family_pool = family_mask & mid_strong_results['remaining_shape_pool']
    n_input = int(family_mask.sum())
    n_pool = int(family_pool.sum())
    expected_shape = SIMPLE_TREE_EXPECTED_CLASS_BY_FAMILY[family_id]
    row = {
        'family_id': family_id,
        'family_name': SIMPLE_TREE_FAMILY_NAMES[family_id],
        'Input': n_input,
        'Eligible shape pool': simple_tree_count_and_percent(n_pool, n_input),
        'Expected shape': expected_shape,
    }
    for label in SIMPLE_TREE_CLASS_ORDER:
        n_label = int((
            family_pool & mid_strong_results['final_shape'].eq(label)
        ).sum())
        row[label] = simple_tree_count_and_percent(n_label, n_pool)
    n_correct = int((
        family_pool
        & mid_strong_results['final_shape'].eq(expected_shape)
    ).sum())
    n_incorrect = n_pool - n_correct
    row['Correct'] = simple_tree_count_and_percent(n_correct, n_pool)
    row['Incorrect'] = simple_tree_count_and_percent(n_incorrect, n_pool)
    row['Accuracy'] = (
        f'{n_correct}/{n_pool} ({n_correct / n_pool:.1%})'
        if n_pool else 'N/A'
    )
    strong_accuracy_step6_rows.append(row)
strong_accuracy_step6 = pd.DataFrame(strong_accuracy_step6_rows)

strong_accuracy_final = mid_final_tables['strong'].copy()

# The older Correct/Incorrect diagnostic tables are intentionally not
# printed. The next block contains the requested row-wise flow tables.
for obsolete_name in [
    'strong_step1_mic_selection_rate.csv',
    'strong_step2_simple_complex_accuracy.csv',
    'strong_step3_strong_monotonicity_accuracy.csv',
    'strong_step4_strong_linearity_accuracy.csv',
    'strong_step5_power_law_linearity_accuracy.csv',
    'strong_step6_shape_accuracy.csv',
    'strong_final_accuracy.csv',
]:
    (MID_TREE_OUTPUT_DIR / obsolete_name).unlink(missing_ok=True)

## Strong only — row-wise distribution table for every step (`0.6 <= MIC < 0.8`)

Each table follows the same format: the first numeric column is the number entering that step (`100%`), and every outcome percentage uses that number as its denominator. Families with zero cases entering a step are omitted from that step's table.

In [22]:
# ============================================================
# STRONG-ONLY FLOW TABLES — COUNTS AND ROW-WISE PERCENTAGES
# ============================================================
def strong_flow_count(n, denominator):
    if denominator == 0:
        return '0 (N/A)'
    return f'{int(n)} ({n / denominator:.1%})'


def strong_flow_distribution_table(
    results,
    eligible_mask,
    eligible_label,
    outcome_masks,
):
    rows = []
    for family_id in SIMPLE_TREE_FAMILY_ORDER:
        family_mask = results['family_id'].eq(family_id)
        family_eligible = family_mask & eligible_mask
        n_eligible = int(family_eligible.sum())
        if n_eligible == 0:
            continue
        row = {
            'family_id': family_id,
            'family_name': SIMPLE_TREE_FAMILY_NAMES[family_id],
            eligible_label: strong_flow_count(n_eligible, n_eligible),
        }
        for outcome_label, outcome_mask in outcome_masks.items():
            n_outcome = int((family_eligible & outcome_mask).sum())
            row[outcome_label] = strong_flow_count(
                n_outcome, n_eligible
            )
        rows.append(row)
    return pd.DataFrame(rows)


# Step 1 uses all three mutually exclusive MIC partitions.
strong_mic_partition_rows = []
strong_mic_partition_counts = {}
for filename, partition_label in [
    ('mic_0_to_0p6_full.parquet', 'MIC < 0.6'),
    ('mic_0p6_to_0p8_full.parquet', '0.6 <= MIC < 0.8'),
    ('mic_0p8_to_1_full.parquet', 'MIC >= 0.8'),
]:
    partition = pd.read_parquet(
        S4_DIR / filename, columns=['family_id', 'variant_level']
    )
    partition = partition.loc[partition['variant_level'].eq('strong')]
    strong_mic_partition_counts[partition_label] = (
        partition.groupby('family_id').size()
    )

for family_id in SIMPLE_TREE_FAMILY_ORDER:
    partition_values = {
        label: int(counts.get(family_id, 0))
        for label, counts in strong_mic_partition_counts.items()
    }
    n_all = sum(partition_values.values())
    if n_all == 0:
        continue
    row = {
        'family_id': family_id,
        'family_name': SIMPLE_TREE_FAMILY_NAMES[family_id],
        'All S4 Strong': strong_flow_count(n_all, n_all),
    }
    for label, value in partition_values.items():
        row[label] = strong_flow_count(value, n_all)
    strong_mic_partition_rows.append(row)
strong_flow_step1 = pd.DataFrame(strong_mic_partition_rows)

# Step 2: every Strong case in the selected MIC interval.
strong_flow_step2 = strong_flow_distribution_table(
    mid_strong_results,
    pd.Series(True, index=mid_strong_results.index),
    'MIC 0.6–0.8 input',
    {
        'Simple': mid_strong_results['simple_relationship'],
        'Complex': ~mid_strong_results['simple_relationship'],
    },
)

# Step 3: only Step-2 Simple cases.
strong_flow_step3 = strong_flow_distribution_table(
    mid_strong_results,
    mid_strong_results['simple_relationship'],
    'Simple',
    {
        'Strong monotonic': mid_strong_results['strong_monotonic'],
        'Not strong monotonic': ~mid_strong_results['strong_monotonic'],
    },
)

# Step 4: every Step-3 Strong-monotonic case receives the power-law fit.
strong_flow_step4 = strong_flow_distribution_table(
    mid_strong_results,
    mid_strong_results['strong_monotonic'],
    'Power-law fit',
    {
        'Linear': mid_strong_results['confirmed_linear'],
        'Non-linear': ~mid_strong_results['confirmed_linear'],
    },
)

# Step 5: all non-linear Simple branches merged together.
strong_flow_step5 = strong_flow_distribution_table(
    mid_strong_results,
    mid_strong_results['remaining_shape_pool'],
    'Cubic polynomial',
    {
        'Concave': mid_strong_results['final_shape'].eq('Concave'),
        'Convex': mid_strong_results['final_shape'].eq('Convex'),
        'S-shaped': mid_strong_results['final_shape'].eq('S-shaped'),
        'Uncertain': mid_strong_results['final_shape'].eq(
            'Other/Uncertain'
        ),
    },
)

# Final result: every Strong case in the selected MIC interval.
strong_flow_final = strong_flow_distribution_table(
    mid_strong_results,
    pd.Series(True, index=mid_strong_results.index),
    'MIC 0.6–0.8 input',
    {
        'Linear': mid_strong_results['final_shape'].eq('Linear'),
        'Concave': mid_strong_results['final_shape'].eq('Concave'),
        'Convex': mid_strong_results['final_shape'].eq('Convex'),
        'S-shaped': mid_strong_results['final_shape'].eq('S-shaped'),
        'Uncertain': mid_strong_results['final_shape'].eq(
            'Other/Uncertain'
        ),
    },
)

for obsolete_flow_name in [
    'strong_flow_step4_strong_linearity.csv',
    'strong_flow_step5_power_law_linearity.csv',
    'strong_flow_step6_cubic_polynomial.csv',
]:
    (MID_TREE_OUTPUT_DIR / obsolete_flow_name).unlink(missing_ok=True)

STRONG_FLOW_TABLES = {
    'step1_mic_partitions': strong_flow_step1,
    'step2_simple_complex': strong_flow_step2,
    'step3_strong_monotonicity': strong_flow_step3,
    'step4_power_law_linearity': strong_flow_step4,
    'step5_cubic_polynomial': strong_flow_step5,
    'final_classification': strong_flow_final,
}

for table_name, table in STRONG_FLOW_TABLES.items():
    table.to_csv(
        MID_TREE_OUTPUT_DIR / f'strong_flow_{table_name}.csv',
        index=False,
    )
    print(f'\nStrong — {table_name.replace("_", " ").title()}')
    display(table)



Strong — Step1 Mic Partitions


,family_id,family_name,All S4 Strong,MIC < 0.6,0.6 <= MIC < 0.8,MIC >= 0.8
0,F01,Linear positive,2270 (100.0%),1121 (49.4%),155 (6.8%),994 (43.8%)
1,F03,Power convex positive,2270 (100.0%),1229 (54.1%),235 (10.4%),806 (35.5%)
2,F05,Power concave positive,2270 (100.0%),1238 (54.5%),176 (7.8%),856 (37.7%)
3,F07,Saturation positive,2270 (100.0%),1633 (71.9%),582 (25.6%),55 (2.4%)
4,F13,S-curve positive,2270 (100.0%),1065 (46.9%),114 (5.0%),1091 (48.1%)
5,F15,Threshold positive,2270 (100.0%),1059 (46.7%),99 (4.4%),1112 (49.0%)
6,F18,U-shape,2270 (100.0%),1181 (52.0%),173 (7.6%),916 (40.4%)
7,F19,Spike,2270 (100.0%),2270 (100.0%),0 (0.0%),0 (0.0%)
8,F21,Cubic,2270 (100.0%),1389 (61.2%),332 (14.6%),549 (24.2%)
9,F22,Oscillation / complex non-monotonic,2270 (100.0%),1130 (49.8%),153 (6.7%),987 (43.5%)



Strong — Step2 Simple Complex


,family_id,family_name,MIC 0.6–0.8 input,Simple,Complex
0,F01,Linear positive,155 (100.0%),155 (100.0%),0 (0.0%)
1,F03,Power convex positive,235 (100.0%),235 (100.0%),0 (0.0%)
2,F05,Power concave positive,176 (100.0%),176 (100.0%),0 (0.0%)
3,F07,Saturation positive,582 (100.0%),0 (0.0%),582 (100.0%)
4,F13,S-curve positive,114 (100.0%),103 (90.4%),11 (9.6%)
5,F15,Threshold positive,99 (100.0%),32 (32.3%),67 (67.7%)
6,F18,U-shape,173 (100.0%),0 (0.0%),173 (100.0%)
7,F21,Cubic,332 (100.0%),209 (63.0%),123 (37.0%)
8,F22,Oscillation / complex non-monotonic,153 (100.0%),0 (0.0%),153 (100.0%)
9,F23,Two Lines,352 (100.0%),0 (0.0%),352 (100.0%)



Strong — Step3 Strong Monotonicity


...

,family_id,family_name,Power-law fit,Linear,Non-linear
0,F01,Linear positive,20 (100.0%),15 (75.0%),5 (25.0%)
1,F05,Power concave positive,45 (100.0%),0 (0.0%),45 (100.0%)



Strong — Step5 Cubic Polynomial


,family_id,family_name,Cubic polynomial,Concave,Convex,S-shaped,Uncertain
0,F01,Linear positive,140 (100.0%),44 (31.4%),28 (20.0%),68 (48.6%),0 (0.0%)
1,F03,Power convex positive,235 (100.0%),0 (0.0%),108 (46.0%),0 (0.0%),127 (54.0%)
2,F05,Power concave positive,176 (100.0%),176 (100.0%),0 (0.0%),0 (0.0%),0 (0.0%)
3,F13,S-curve positive,103 (100.0%),0 (0.0%),0 (0.0%),25 (24.3%),78 (75.7%)
4,F15,Threshold positive,32 (100.0%),0 (0.0%),0 (0.0%),4 (12.5%),28 (87.5%)
5,F21,Cubic,209 (100.0%),0 (0.0%),0 (0.0%),0 (0.0%),209 (100.0%)



Strong — Final Classification


,family_id,family_name,MIC 0.6–0.8 input,Linear,Concave,Convex,S-shaped,Uncertain
0,F01,Linear positive,155 (100.0%),15 (9.7%),44 (28.4%),28 (18.1%),68 (43.9%),0 (0.0%)
1,F03,Power convex positive,235 (100.0%),0 (0.0%),0 (0.0%),108 (46.0%),0 (0.0%),127 (54.0%)
2,F05,Power concave positive,176 (100.0%),0 (0.0%),176 (100.0%),0 (0.0%),0 (0.0%),0 (0.0%)
3,F07,Saturation positive,582 (100.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),582 (100.0%)
4,F13,S-curve positive,114 (100.0%),0 (0.0%),0 (0.0%),0 (0.0%),25 (21.9%),89 (78.1%)
5,F15,Threshold positive,99 (100.0%),0 (0.0%),0 (0.0%),0 (0.0%),4 (4.0%),95 (96.0%)
6,F18,U-shape,173 (100.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),173 (100.0%)
7,F21,Cubic,332 (100.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),332 (100.0%)
8,F22,Oscillation / complex non-monotonic,153 (100.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),153 (100.0%)
9,F23,Two Lines,352 (100.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),352 (100.0%)


## Strong only — compact final table for the six target families

This table uses the original `0.6 <= MIC < 0.8` family count as the row denominator. `Linear` is the final confirmed Linear output; it is not the number entering the cubic-polynomial step.

In [23]:
STRONG_COMPACT_FAMILIES = ['F01', 'F03', 'F05', 'F13', 'F15', 'F21']
strong_compact_final_rows = []
for family_id in STRONG_COMPACT_FAMILIES:
    family_mask = mid_strong_results['family_id'].eq(family_id)
    n_input = int(family_mask.sum())
    expected_shape = SIMPLE_TREE_EXPECTED_CLASS_BY_FAMILY[family_id]
    row = {
        'family_id': family_id,
        'family_name': SIMPLE_TREE_FAMILY_NAMES[family_id],
        'MIC 0.6–0.8 input': n_input,
    }
    for display_label, result_label in [
        ('Linear', 'Linear'),
        ('Concave', 'Concave'),
        ('Convex', 'Convex'),
        ('S-shaped', 'S-shaped'),
        ('Other / Uncertain', 'Other/Uncertain'),
    ]:
        n_label = int((
            family_mask & mid_strong_results['final_shape'].eq(result_label)
        ).sum())
        row[display_label] = strong_flow_count(n_label, n_input)
    n_correct = int((
        family_mask & mid_strong_results['final_shape'].eq(expected_shape)
    ).sum())
    row['Accuracy'] = (
        f'{n_correct}/{n_input} ({n_correct / n_input:.1%})'
        if n_input else 'N/A'
    )
    strong_compact_final_rows.append(row)

strong_compact_final_table = pd.DataFrame(strong_compact_final_rows)
strong_compact_final_table.to_csv(
    MID_TREE_OUTPUT_DIR / 'strong_compact_final_six_families.csv',
    index=False,
)
display(strong_compact_final_table)

# F01 cascade identifies where Linear cases are lost.
f01_mask = mid_strong_results['family_id'].eq('F01')
f01_total = int(f01_mask.sum())
f01_cascade = pd.DataFrame([
    {
        'Stage': 'MIC 0.6–0.8 input',
        'F01 retained': strong_flow_count(f01_total, f01_total),
    },
    {
        'Stage': 'Simple',
        'F01 retained': strong_flow_count(
            int((f01_mask & mid_strong_results['simple_relationship']).sum()),
            f01_total,
        ),
    },
    {
        'Stage': 'Strong monotonic',
        'F01 retained': strong_flow_count(
            int((f01_mask & mid_strong_results['strong_monotonic']).sum()),
            f01_total,
        ),
    },
    {
        'Stage': 'Strong linearity',
        'F01 retained': strong_flow_count(
            int((
                f01_mask
                & mid_strong_results['strong_linearity_candidate']
            ).sum()),
            f01_total,
        ),
    },
    {
        'Stage': 'Power-law confirmed Linear',
        'F01 retained': strong_flow_count(
            int((f01_mask & mid_strong_results['confirmed_linear']).sum()),
            f01_total,
        ),
    },
])
f01_cascade.to_csv(
    MID_TREE_OUTPUT_DIR / 'strong_f01_linear_cascade.csv', index=False
)
display(f01_cascade)

,family_id,family_name,MIC 0.6–0.8 input,Linear,Concave,Convex,S-shaped,Other / Uncertain,Accuracy
0,F01,Linear positive,155,15 (9.7%),44 (28.4%),28 (18.1%),68 (43.9%),0 (0.0%),15/155 (9.7%)
1,F03,Power convex positive,235,0 (0.0%),0 (0.0%),108 (46.0%),0 (0.0%),127 (54.0%),108/235 (46.0%)
2,F05,Power concave positive,176,0 (0.0%),176 (100.0%),0 (0.0%),0 (0.0%),0 (0.0%),176/176 (100.0%)
3,F13,S-curve positive,114,0 (0.0%),0 (0.0%),0 (0.0%),25 (21.9%),89 (78.1%),25/114 (21.9%)
4,F15,Threshold positive,99,0 (0.0%),0 (0.0%),0 (0.0%),4 (4.0%),95 (96.0%),95/99 (96.0%)
5,F21,Cubic,332,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),332 (100.0%),332/332 (100.0%)


,Stage,F01 retained
0,MIC 0.6–0.8 input,155 (100.0%)
1,Simple,155 (100.0%)
2,Strong monotonic,20 (12.9%)
3,Strong linearity,20 (12.9%)
4,Power-law confirmed Linear,15 (9.7%)
